# Home Credit Default Risk
## End-to-End ML Project · Headroom Dataset for LLM Evaluation

**Goal 1 (DS Project):** Build an industry-level credit risk model — data loading → EDA → preprocessing → feature engineering → modeling → evaluation.

**Goal 2 (Headroom Dataset):** At each pipeline stage, log prompts where *Gemini Pro* is expected to fail or give a wrong/incomplete answer as a data scientist. Each logged task captures `stage`, `prompt`, `context`, `expected_answer`, `failure_category`, and `difficulty`.

**Dataset:** [Home Credit Default Risk](https://www.kaggle.com/competitions/home-credit-default-risk) (Kaggle)

| Table | Rows | Description |
|---|---|---|
| application_train/test | 307k / 48k | Main application features + target |
| bureau | 1.7M | Credit bureau history (one row per external loan) |
| bureau_balance | 27M | Monthly snapshots of bureau loans |
| previous_application | 1.7M | Previous Home Credit loan applications |
| POS_CASH_balance | 10M | Monthly POS/cash loan balance snapshots |
| installments_payments | 13M | Installment payment history |
| credit_card_balance | 3.8M | Monthly credit card balance snapshots |

---

## 1. Environment Setup

In [ ]:
%%capture
!pip install lightgbm xgboost shap imbalanced-learn optuna -q

### Dataset Download via Google Drive

**Steps:**
1. Go to [kaggle.com/competitions/home-credit-default-risk/data](https://www.kaggle.com/competitions/home-credit-default-risk/data)
2. Click **Download All** → downloads `home-credit-default-risk.zip` (~300MB)
3. Upload the zip file to your **Google Drive** (anywhere, e.g. `My Drive/`)
4. Update `ZIP_PATH` in the cell below to match where you uploaded it
5. Run the cell — it will mount Drive, unzip, and set up `/content/data/`

In [ ]:
from google.colab import drive
import zipfile, os

# ── Mount Google Drive ─────────────────────────────────────────────────────────
drive.mount('/content/drive')

# ── Update this path to where you uploaded the zip in Drive ───────────────────
ZIP_PATH = '/content/drive/MyDrive/home-credit-default-risk.zip'

DATA_DIR = '/content/data'
OUT_DIR  = '/content/outputs'
os.makedirs(DATA_DIR, exist_ok=True)
os.makedirs(OUT_DIR,  exist_ok=True)

# ── Extract ────────────────────────────────────────────────────────────────────
print('Extracting dataset...')
with zipfile.ZipFile(ZIP_PATH, 'r') as z:
    z.extractall(DATA_DIR)

print('Done. Files in /content/data:')
print(os.listdir(DATA_DIR))

In [ ]:
import os, gc, warnings, json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
from scipy import stats
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import StratifiedKFold
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    roc_auc_score, roc_curve, average_precision_score,
    precision_recall_curve, confusion_matrix, classification_report
)
from sklearn.feature_selection import VarianceThreshold
import lightgbm as lgb
import xgboost as xgb
import shap

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', 100)
pd.set_option('display.float_format', '{:.4f}'.format)

# ── Global config ──────────────────────────────────────────────────────────────
DATA_DIR = '/content/data'
OUT_DIR  = '/content/outputs'
SEED     = 42
N_FOLDS  = 5
TARGET   = 'TARGET'
np.random.seed(SEED)

print('Libraries loaded  ✓')

In [ ]:
class HeadroomLogger:
    '''
    Captures DS workflow tasks where Gemini Pro fails acting as a data scientist.
    Tasks are grounded in REAL pipeline outputs — not hand-written descriptions.
    Ground truth is derived from actually running the ML pipeline (LightGBM etc.).
    '''

    def __init__(self):
        self.tasks = []
        self._counter = 0

    def log(self, stage: str, prompt: str, data_context: str,
             ground_truth: str, failure_category: str,
             difficulty: str = 'hard') -> None:
        self._counter += 1
        self.tasks.append({
            'task_id':          f'HT-{self._counter:03d}',
            'stage':            stage,
            'prompt':           prompt,
            'data_context':     data_context,    # real pipeline outputs shown to Gemini
            'ground_truth':     ground_truth,    # correct answer derived from pipeline
            'failure_category': failure_category,
            'difficulty':       difficulty,
            'dataset':          'Home Credit Default Risk',
            'gemini_model_target': 'gemini-2.0-pro',
            'gemini_response':  None             # filled after Gemini is called
        })
        print(f'[HeadroomLogger] Logged HT-{self._counter:03d}  stage={stage}  '
              f'category={failure_category}')

    def to_dataframe(self) -> pd.DataFrame:
        return pd.DataFrame(self.tasks)

    def export(self, base_path: str = '/content/outputs/headroom_dataset') -> None:
        df = self.to_dataframe()
        df.to_csv(f'{base_path}.csv', index=False)
        df.to_json(f'{base_path}.json', orient='records', indent=2)
        print(f'Headroom dataset exported: {len(df)} tasks')
        display(df[['task_id', 'stage', 'failure_category', 'difficulty']])


logger = HeadroomLogger()

# Ground truth dict — populated throughout the pipeline, used to create tasks in Phase 2
ground_truth = {}

print('HeadroomLogger and ground_truth dict initialized  ✓')

---
## 2. Data Loading & Overview

In [ ]:
def load_data(data_dir: str) -> dict:
    '''Load all 7 Home Credit tables.'''
    tables = {
        'app_train':    'application_train.csv',
        'app_test':     'application_test.csv',
        'bureau':       'bureau.csv',
        'bureau_bal':   'bureau_balance.csv',
        'prev_app':     'previous_application.csv',
        'pos_cash':     'POS_CASH_balance.csv',
        'installments': 'installments_payments.csv',
        'credit_card':  'credit_card_balance.csv',
    }
    data = {}
    for key, fname in tables.items():
        path = os.path.join(data_dir, fname)
        data[key] = pd.read_csv(path)
        print(f'{key:15s}  shape: {str(data[key].shape):20s}  '
              f'memory: {data[key].memory_usage(deep=True).sum() / 1e6:.1f} MB')
    return data


data = load_data(DATA_DIR)
app_train = data['app_train']
app_test  = data['app_test']

print(f'\nTotal training samples : {len(app_train):,}')
print(f'Total test samples     : {len(app_test):,}')
print(f'Number of features     : {app_train.shape[1]}')
print(f'Target positive rate   : {app_train[TARGET].mean():.2%}')
print(f'\nClass counts:')
print(app_train[TARGET].value_counts())

---
## 3. Exploratory Data Analysis (EDA)

In [ ]:
# ── Target distribution ────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

counts = app_train[TARGET].value_counts()
colors = ['#2ECC71', '#E74C3C']

axes[0].pie(
    counts, labels=['No Default (0)', 'Default (1)'],
    autopct='%1.1f%%', colors=colors, startangle=90,
    wedgeprops={'edgecolor': 'white', 'linewidth': 2}
)
axes[0].set_title('Target Distribution', fontsize=14, fontweight='bold')

sns.barplot(
    x=counts.index.map({0: 'No Default', 1: 'Default'}),
    y=counts.values, palette=colors, ax=axes[1]
)
axes[1].set_title('Class Counts', fontsize=14, fontweight='bold')
axes[1].set_ylabel('Count')
for i, v in enumerate(counts.values):
    axes[1].text(i, v + 500, f'{v:,}', ha='center', fontweight='bold')

plt.suptitle('Target Variable Analysis', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.savefig(f'{OUT_DIR}/01_target_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

print(f'Class imbalance ratio : {counts[0]/counts[1]:.1f}:1')
print(f'Strategy note: extreme imbalance requires careful metric choice and threshold calibration.')

In [ ]:
def plot_missing(df: pd.DataFrame, title: str, top_n: int = 40) -> pd.DataFrame:
    '''Visualize top-N features by missing value percentage.'''
    missing_pct = (df.isnull().mean() * 100).sort_values(ascending=False)
    missing_pct = missing_pct[missing_pct > 0].head(top_n)

    if len(missing_pct) == 0:
        print(f'{title}: No missing values.')
        return pd.DataFrame()

    colors = ['#E74C3C' if x > 50 else '#F39C12' if x > 20 else '#3498DB'
              for x in missing_pct.values]

    fig, ax = plt.subplots(figsize=(13, max(6, len(missing_pct) * 0.28)))
    ax.barh(missing_pct.index[::-1], missing_pct.values[::-1],
            color=colors[::-1], edgecolor='white')
    ax.set_xlabel('Missing (%)', fontsize=11)
    ax.set_title(f'{title} — Missing Value Analysis (Top {top_n})',
                 fontsize=13, fontweight='bold')
    ax.axvline(x=50, color='red', linestyle='--', alpha=0.7, label='>50% threshold')
    ax.axvline(x=20, color='orange', linestyle='--', alpha=0.7, label='>20% threshold')
    ax.legend(fontsize=9)
    plt.tight_layout()
    plt.savefig(f'{OUT_DIR}/02_missing_{title.lower().replace(" ","_")}.png',
                dpi=150, bbox_inches='tight')
    plt.show()
    return pd.DataFrame({'feature': missing_pct.index, 'missing_pct': missing_pct.values})


missing_df = plot_missing(app_train, 'Application Train')
print(f'Features with >50% missing : {(missing_df.missing_pct > 50).sum()}')
print(f'Features with >20% missing : {(missing_df.missing_pct > 20).sum()}')

# ── Populate ground truth ──────────────────────────────────────────────────────
ground_truth['missing_top10'] = missing_df.head(10)[['feature', 'missing_pct']].to_dict('records')
ground_truth['missing_over50_count'] = int((missing_df.missing_pct > 50).sum())
ground_truth['missing_over20_count'] = int((missing_df.missing_pct > 20).sum())

In [ ]:
# ── Numerical feature distributions vs target ──────────────────────────────────
key_num_cols = [
    'AMT_INCOME_TOTAL', 'AMT_CREDIT', 'AMT_ANNUITY', 'AMT_GOODS_PRICE',
    'DAYS_BIRTH', 'DAYS_EMPLOYED', 'EXT_SOURCE_1', 'EXT_SOURCE_2', 'EXT_SOURCE_3'
]

fig, axes = plt.subplots(3, 3, figsize=(16, 12))
axes = axes.flatten()

for i, col in enumerate(key_num_cols):
    if col not in app_train.columns:
        continue
    for label, color in zip([0, 1], ['#2ECC71', '#E74C3C']):
        subset = app_train.loc[app_train[TARGET] == label, col].dropna()
        axes[i].hist(subset, bins=50, alpha=0.5, color=color,
                     label=f'Target={label}', density=True)
    axes[i].set_title(col, fontsize=10, fontweight='bold')
    axes[i].legend(fontsize=8)
    axes[i].set_ylabel('Density')

plt.suptitle('Key Numerical Features vs Target', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(f'{OUT_DIR}/03_numerical_distributions.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── Categorical feature default rates ──────────────────────────────────────────
key_cat_cols = [
    'NAME_CONTRACT_TYPE', 'CODE_GENDER', 'NAME_INCOME_TYPE',
    'NAME_EDUCATION_TYPE', 'NAME_FAMILY_STATUS', 'OCCUPATION_TYPE'
]

fig, axes = plt.subplots(2, 3, figsize=(18, 10))
axes = axes.flatten()

for i, col in enumerate(key_cat_cols):
    if col not in app_train.columns:
        continue
    default_rate = app_train.groupby(col)[TARGET].mean().sort_values(ascending=False)
    counts_by_cat = app_train[col].value_counts()
    # Filter to categories with at least 100 samples
    valid = counts_by_cat[counts_by_cat >= 100].index
    default_rate = default_rate.loc[default_rate.index.isin(valid)]

    colors_bar = plt.cm.RdYlGn_r(np.linspace(0.2, 0.8, len(default_rate)))
    axes[i].bar(range(len(default_rate)), default_rate.values * 100, color=colors_bar)
    axes[i].set_xticks(range(len(default_rate)))
    axes[i].set_xticklabels(default_rate.index, rotation=35, ha='right', fontsize=8)
    axes[i].set_title(f'{col}', fontsize=10, fontweight='bold')
    axes[i].set_ylabel('Default Rate (%)')
    axes[i].axhline(y=app_train[TARGET].mean() * 100, color='black',
                    linestyle='--', alpha=0.6, label='Overall rate')
    axes[i].legend(fontsize=8)

plt.suptitle('Default Rate by Categorical Feature', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(f'{OUT_DIR}/04_categorical_default_rates.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── Secondary tables overview ──────────────────────────────────────────────────
print('=== Secondary Table Statistics ===')

for name, key in [('Bureau', 'bureau'), ('Bureau Balance', 'bureau_bal'),
                   ('Previous Application', 'prev_app'), ('POS Cash', 'pos_cash'),
                   ('Installments', 'installments'), ('Credit Card', 'credit_card')]:
    df = data[key]
    id_col = 'SK_ID_CURR' if 'SK_ID_CURR' in df.columns else df.columns[0]
    n_unique = df['SK_ID_CURR'].nunique() if 'SK_ID_CURR' in df.columns else 'N/A'
    print(f'\n{name}:')
    print(f'  Rows        : {len(df):,}')
    print(f'  Unique IDs  : {n_unique:,}' if isinstance(n_unique, int) else f'  Unique IDs  : {n_unique}')
    print(f'  Avg rows/ID : {len(df)/n_unique:.1f}' if isinstance(n_unique, int) else '')
    print(f'  Missing %   : {df.isnull().mean().mean():.2%}')

---
## 4. Preprocessing

In [ ]:
def preprocess_application(df: pd.DataFrame) -> pd.DataFrame:
    '''
    Industry-level preprocessing for the application table.

    Key decisions:
    1. DAYS_EMPLOYED = 365243 is a domain-specific placeholder (retirees/unemployed)
    2. XNA in categorical columns treated as missing
    3. Binary categoricals label-encoded; multi-class one-hot encoded
    4. Numeric missing values imputed with column median
    5. AGE_YEARS derived as interpretable feature from DAYS_BIRTH
    '''
    df = df.copy()

    # ── 1. DAYS_EMPLOYED anomaly: 365243 = retired / unemployed / pensioner ──
    df['DAYS_EMPLOYED_ANOM'] = (df['DAYS_EMPLOYED'] == 365243).astype(np.int8)
    df['DAYS_EMPLOYED'] = df['DAYS_EMPLOYED'].replace(365243, np.nan)
    df['DAYS_EMPLOYED'].fillna(df['DAYS_EMPLOYED'].median(), inplace=True)

    # ── 2. Replace XNA (explicit missing code) ──────────────────────────────
    df.replace('XNA', np.nan, inplace=True)

    # ── 3. Derived time features ────────────────────────────────────────────
    df['AGE_YEARS'] = (-df['DAYS_BIRTH'] / 365).astype(int)
    df['YEARS_EMPLOYED'] = (-df['DAYS_EMPLOYED'] / 365).clip(lower=0)
    df['YEARS_ID_PUBLISH'] = -df['DAYS_ID_PUBLISH'] / 365
    df['YEARS_REGISTRATION'] = -df['DAYS_REGISTRATION'] / 365
    df['YEARS_LAST_PHONE_CHANGE'] = -df['DAYS_LAST_PHONE_CHANGE'] / 365

    # ── 4. Encode binary categoricals ───────────────────────────────────────
    binary_cols = [c for c in df.select_dtypes('object').columns
                   if df[c].nunique() <= 2]
    le = LabelEncoder()
    for col in binary_cols:
        df[col] = le.fit_transform(df[col].astype(str))

    # ── 5. One-hot encode remaining categoricals ─────────────────────────────
    df = pd.get_dummies(df, dummy_na=False)

    # ── 6. Impute remaining numeric missing values ───────────────────────────
    num_cols = df.select_dtypes(include=[np.number]).columns
    df[num_cols] = df[num_cols].fillna(df[num_cols].median())

    return df


app_train_proc = preprocess_application(app_train)
app_test_proc  = preprocess_application(app_test)

# Align columns: test may lack some dummies created from train
app_train_proc, app_test_proc = app_train_proc.align(
    app_test_proc, join='left', axis=1, fill_value=0
)

print(f'Processed train shape : {app_train_proc.shape}')
print(f'Processed test shape  : {app_test_proc.shape}')
print(f'DAYS_EMPLOYED anomaly : {app_train_proc["DAYS_EMPLOYED_ANOM"].sum():,} rows ({app_train_proc["DAYS_EMPLOYED_ANOM"].mean():.1%})')

# ── Populate ground truth ──────────────────────────────────────────────────────
ground_truth['days_employed_anom_count'] = int(app_train_proc['DAYS_EMPLOYED_ANOM'].sum())
ground_truth['days_employed_anom_pct']   = float(app_train_proc['DAYS_EMPLOYED_ANOM'].mean())
ground_truth['days_employed_non_anom_median'] = float(
    app_train.loc[app_train['DAYS_EMPLOYED'] != 365243, 'DAYS_EMPLOYED'].median()
)

---
## 5. Feature Engineering

All secondary tables are aggregated to `SK_ID_CURR` level before joining to the main table.
The aggregation strategy at each step is intentionally non-trivial — this is a primary source of headroom tasks.

In [ ]:
def aggregate_bureau(bureau: pd.DataFrame, bureau_bal: pd.DataFrame) -> pd.DataFrame:
    '''
    Two-stage aggregation:
      Stage 1: bureau_balance (monthly snapshots) → loan level (SK_ID_BUREAU)
      Stage 2: loan level → applicant level (SK_ID_CURR)

    Key design choices:
    - Distinguish Active vs Closed loans (different default predictability)
    - Compute DPD ratio from monthly STATUS codes (domain: 0=no DPD, 1-5=DPD days)
    - Capture credit history length (DAYS_CREDIT oldest entry)
    - Overdue amount features (AMT_CREDIT_SUM_OVERDUE) are highly predictive
    '''

    # Stage 1: bureau_balance → loan-level aggregations ──────────────────────
    bbal_agg = bureau_bal.groupby('SK_ID_BUREAU').agg(
        BBAL_MONTHS_MIN   = ('MONTHS_BALANCE', 'min'),
        BBAL_MONTHS_MAX   = ('MONTHS_BALANCE', 'max'),
        BBAL_TOTAL_MONTHS = ('MONTHS_BALANCE', 'count'),
        BBAL_STATUS_C     = ('STATUS', lambda x: (x == 'C').sum()),
        BBAL_STATUS_X     = ('STATUS', lambda x: (x == 'X').sum()),
        BBAL_STATUS_0     = ('STATUS', lambda x: (x == '0').sum()),
        BBAL_STATUS_1     = ('STATUS', lambda x: (x == '1').sum()),
        BBAL_STATUS_2     = ('STATUS', lambda x: (x == '2').sum()),
        BBAL_STATUS_3     = ('STATUS', lambda x: (x == '3').sum()),
        BBAL_STATUS_4     = ('STATUS', lambda x: (x == '4').sum()),
        BBAL_STATUS_5     = ('STATUS', lambda x: (x == '5').sum()),
    ).reset_index()

    # DPD ratio: proportion of months with any days-past-due
    dpd_cols = ['BBAL_STATUS_1', 'BBAL_STATUS_2', 'BBAL_STATUS_3',
                'BBAL_STATUS_4', 'BBAL_STATUS_5']
    bbal_agg['BBAL_DPD_MONTHS'] = bbal_agg[dpd_cols].sum(axis=1)
    bbal_agg['BBAL_DPD_RATIO']  = (
        bbal_agg['BBAL_DPD_MONTHS'] / (bbal_agg['BBAL_TOTAL_MONTHS'] + 1e-8)
    )

    # Stage 2: Enrich bureau with balance features, then aggregate to customer level
    bureau_enr = bureau.merge(bbal_agg, on='SK_ID_BUREAU', how='left')

    num_cols = [c for c in bureau_enr.select_dtypes(include=[np.number]).columns
                if c not in ('SK_ID_CURR', 'SK_ID_BUREAU')]

    agg_spec = {}
    for col in num_cols:
        agg_spec[f'BUR_{col}_MEAN'] = (col, 'mean')
        agg_spec[f'BUR_{col}_MAX']  = (col, 'max')
        agg_spec[f'BUR_{col}_SUM']  = (col, 'sum')

    agg_spec['BUR_LOAN_COUNT']    = ('SK_ID_BUREAU', 'count')
    agg_spec['BUR_ACTIVE_LOANS']  = ('CREDIT_ACTIVE', lambda x: (x == 'Active').sum())
    agg_spec['BUR_CLOSED_LOANS']  = ('CREDIT_ACTIVE', lambda x: (x == 'Closed').sum())
    agg_spec['BUR_ACTIVE_RATIO']  = ('CREDIT_ACTIVE', lambda x: (x == 'Active').mean())
    agg_spec['BUR_OVERDUE_SUM']   = ('AMT_CREDIT_SUM_OVERDUE', 'sum')
    agg_spec['BUR_OVERDUE_MAX']   = ('AMT_CREDIT_SUM_OVERDUE', 'max')
    agg_spec['BUR_CREDIT_HIST_DAYS'] = ('DAYS_CREDIT', 'min')  # Most distant = oldest

    bureau_feat = bureau_enr.groupby('SK_ID_CURR').agg(**agg_spec).reset_index()

    print(f'Bureau features: {bureau_feat.shape}')
    return bureau_feat


bureau_feat = aggregate_bureau(data['bureau'], data['bureau_bal'])

In [ ]:
def aggregate_previous_application(prev: pd.DataFrame) -> pd.DataFrame:
    '''Aggregate previous Home Credit applications to SK_ID_CURR level.'''
    prev = prev.copy()
    prev.replace('XNA', np.nan, inplace=True)

    # Fix same 365243 anomaly in date columns
    for col in ('DAYS_FIRST_DRAWING', 'DAYS_LAST_DUE', 'DAYS_TERMINATION'):
        if col in prev.columns:
            prev[col].replace(365243, np.nan, inplace=True)

    # Loan-level derived features
    prev['PREV_APP_CREDIT_RATIO']  = prev['AMT_APPLICATION'] / (prev['AMT_CREDIT'] + 1e-8)
    prev['PREV_CREDIT_GOODS_RATIO'] = prev['AMT_CREDIT'] / (prev['AMT_GOODS_PRICE'] + 1e-8)
    prev['PREV_DOWN_PAYMENT_RATIO'] = prev['AMT_DOWN_PAYMENT'] / (prev['AMT_CREDIT'] + 1e-8)

    num_cols = [c for c in prev.select_dtypes(include=[np.number]).columns
                if c not in ('SK_ID_CURR', 'SK_ID_PREV')]

    agg_spec = {f'PREV_{c}_{a}': (c, a)
                for c in num_cols for a in ('mean', 'max', 'min')}

    agg_spec['PREV_LOAN_COUNT']     = ('SK_ID_PREV', 'count')
    agg_spec['PREV_APPROVED']       = ('NAME_CONTRACT_STATUS', lambda x: (x == 'Approved').sum())
    agg_spec['PREV_REFUSED']        = ('NAME_CONTRACT_STATUS', lambda x: (x == 'Refused').sum())
    agg_spec['PREV_APPROVAL_RATE']  = ('NAME_CONTRACT_STATUS', lambda x: (x == 'Approved').mean())
    agg_spec['PREV_CANCELLED']      = ('NAME_CONTRACT_STATUS', lambda x: (x == 'Canceled').sum())

    prev_feat = prev.groupby('SK_ID_CURR').agg(**agg_spec).reset_index()
    print(f'Previous application features: {prev_feat.shape}')
    return prev_feat


prev_feat = aggregate_previous_application(data['prev_app'])

In [ ]:
def aggregate_pos_cash(pos: pd.DataFrame) -> pd.DataFrame:
    '''Aggregate POS/cash loan monthly balances to SK_ID_CURR level.'''
    pos = pos.copy()

    # DPD rate at installment level
    pos['POS_DPD_FLAG']    = (pos['SK_DPD'] > 0).astype(np.int8)
    pos['POS_DPD_DEF_FLAG'] = (pos['SK_DPD_DEF'] > 0).astype(np.int8)

    agg_spec = {
        'POS_MONTHS_COUNT':      ('MONTHS_BALANCE', 'count'),
        'POS_MONTHS_MIN':        ('MONTHS_BALANCE', 'min'),
        'POS_SK_DPD_MEAN':       ('SK_DPD', 'mean'),
        'POS_SK_DPD_MAX':        ('SK_DPD', 'max'),
        'POS_SK_DPD_SUM':        ('SK_DPD', 'sum'),
        'POS_SK_DPD_DEF_MEAN':   ('SK_DPD_DEF', 'mean'),
        'POS_SK_DPD_DEF_MAX':    ('SK_DPD_DEF', 'max'),
        'POS_DPD_RATE':          ('POS_DPD_FLAG', 'mean'),
        'POS_DPD_DEF_RATE':      ('POS_DPD_DEF_FLAG', 'mean'),
        'POS_CNT_INST_MEAN':     ('CNT_INSTALMENT', 'mean'),
        'POS_CNT_INST_FUTURE_MEAN': ('CNT_INSTALMENT_FUTURE', 'mean'),
        'POS_LOAN_COUNT':        ('SK_ID_PREV', 'nunique'),
        'POS_COMPLETED':         ('NAME_CONTRACT_STATUS', lambda x: (x == 'Completed').sum()),
        'POS_ACTIVE':            ('NAME_CONTRACT_STATUS', lambda x: (x == 'Active').sum()),
    }
    pos_feat = pos.groupby('SK_ID_CURR').agg(**agg_spec).reset_index()
    print(f'POS Cash features: {pos_feat.shape}')
    return pos_feat


pos_feat = aggregate_pos_cash(data['pos_cash'])

In [ ]:
def aggregate_installments(inst: pd.DataFrame) -> pd.DataFrame:
    '''Aggregate installment payment history to SK_ID_CURR level.'''
    inst = inst.copy()

    # Payment quality metrics
    inst['INST_PAYMENT_DIFF']  = inst['AMT_INSTALMENT'] - inst['AMT_PAYMENT']
    inst['INST_PAYMENT_RATIO'] = inst['AMT_PAYMENT'] / (inst['AMT_INSTALMENT'] + 1e-8)
    inst['INST_DAYS_LATE']     = (inst['DAYS_ENTRY_PAYMENT'] - inst['DAYS_INSTALMENT']).clip(lower=0)
    inst['INST_IS_LATE']       = (inst['INST_DAYS_LATE'] > 0).astype(np.int8)
    inst['INST_IS_UNDERPAID']  = (inst['INST_PAYMENT_DIFF'] > 0).astype(np.int8)

    agg_spec = {
        'INST_PAYMENT_DIFF_MEAN':  ('INST_PAYMENT_DIFF',  'mean'),
        'INST_PAYMENT_DIFF_MAX':   ('INST_PAYMENT_DIFF',  'max'),
        'INST_PAYMENT_DIFF_SUM':   ('INST_PAYMENT_DIFF',  'sum'),
        'INST_PAYMENT_RATIO_MEAN': ('INST_PAYMENT_RATIO', 'mean'),
        'INST_PAYMENT_RATIO_MIN':  ('INST_PAYMENT_RATIO', 'min'),
        'INST_DAYS_LATE_MEAN':     ('INST_DAYS_LATE',     'mean'),
        'INST_DAYS_LATE_MAX':      ('INST_DAYS_LATE',     'max'),
        'INST_DAYS_LATE_SUM':      ('INST_DAYS_LATE',     'sum'),
        'INST_LATE_RATE':          ('INST_IS_LATE',       'mean'),
        'INST_LATE_COUNT':         ('INST_IS_LATE',       'sum'),
        'INST_UNDERPAID_RATE':     ('INST_IS_UNDERPAID',  'mean'),
        'INST_LOAN_COUNT':         ('SK_ID_PREV',         'nunique'),
        'INST_COUNT':              ('SK_ID_PREV',         'count'),
        'INST_AMT_INSTALMENT_SUM': ('AMT_INSTALMENT',     'sum'),
        'INST_AMT_PAYMENT_SUM':    ('AMT_PAYMENT',        'sum'),
    }
    inst_feat = inst.groupby('SK_ID_CURR').agg(**agg_spec).reset_index()
    print(f'Installment features: {inst_feat.shape}')
    return inst_feat


inst_feat = aggregate_installments(data['installments'])

In [ ]:
def aggregate_credit_card(cc: pd.DataFrame) -> pd.DataFrame:
    '''Aggregate credit card balance history to SK_ID_CURR level.'''
    cc = cc.copy()
    cc.replace('XNA', np.nan, inplace=True)

    # Credit utilization (over-limit is a strong default signal)
    cc['CC_UTILIZATION']       = cc['AMT_BALANCE'] / (cc['AMT_CREDIT_LIMIT_ACTUAL'] + 1e-8)
    cc['CC_OVERLIMIT']         = (cc['CC_UTILIZATION'] > 1.0).astype(np.int8)
    cc['CC_PAYMENT_RATIO']     = cc['AMT_PAYMENT_CURRENT'] / (cc['AMT_INST_MIN_REGULARITY'] + 1e-8)
    cc['CC_MIN_PAY_MISSED']    = (cc['AMT_PAYMENT_CURRENT'] < cc['AMT_INST_MIN_REGULARITY']).astype(np.int8)

    agg_spec = {
        'CC_BALANCE_MEAN':        ('AMT_BALANCE',           'mean'),
        'CC_BALANCE_MAX':         ('AMT_BALANCE',           'max'),
        'CC_BALANCE_SUM':         ('AMT_BALANCE',           'sum'),
        'CC_UTILIZATION_MEAN':    ('CC_UTILIZATION',        'mean'),
        'CC_UTILIZATION_MAX':     ('CC_UTILIZATION',        'max'),
        'CC_OVERLIMIT_COUNT':     ('CC_OVERLIMIT',          'sum'),
        'CC_OVERLIMIT_RATE':      ('CC_OVERLIMIT',          'mean'),
        'CC_MIN_PAY_MISSED_RATE': ('CC_MIN_PAY_MISSED',     'mean'),
        'CC_MIN_PAY_MISSED_COUNT':('CC_MIN_PAY_MISSED',     'sum'),
        'CC_DPD_MAX':             ('SK_DPD',                'max'),
        'CC_DPD_MEAN':            ('SK_DPD',                'mean'),
        'CC_DPD_SUM':             ('SK_DPD',                'sum'),
        'CC_LOAN_COUNT':          ('SK_ID_PREV',            'nunique'),
        'CC_MONTHS_COUNT':        ('MONTHS_BALANCE',        'count'),
        'CC_DRAWINGS_MEAN':       ('AMT_DRAWINGS_CURRENT',  'mean'),
        'CC_ATM_DRAWINGS_MEAN':   ('AMT_DRAWINGS_ATM_CURRENT', 'mean'),
    }
    cc_feat = cc.groupby('SK_ID_CURR').agg(**agg_spec).reset_index()
    print(f'Credit card features: {cc_feat.shape}')
    return cc_feat


cc_feat = aggregate_credit_card(data['credit_card'])

In [ ]:
def engineer_domain_features(df: pd.DataFrame) -> pd.DataFrame:
    '''
    Credit-domain derived features.
    These capture economic relationships that raw variables miss.
    Ratios and interactions are standard in credit scorecard development.
    '''
    df = df.copy()

    # ── Credit-to-income ratios (core underwriting metrics) ─────────────────
    df['CREDIT_INCOME_RATIO']   = df['AMT_CREDIT']      / (df['AMT_INCOME_TOTAL'] + 1e-8)
    df['ANNUITY_INCOME_RATIO']  = df['AMT_ANNUITY']     / (df['AMT_INCOME_TOTAL'] + 1e-8)
    df['GOODS_INCOME_RATIO']    = df['AMT_GOODS_PRICE'] / (df['AMT_INCOME_TOTAL'] + 1e-8)
    df['CREDIT_GOODS_RATIO']    = df['AMT_CREDIT']      / (df['AMT_GOODS_PRICE'] + 1e-8)
    df['ANNUITY_CREDIT_RATIO']  = df['AMT_ANNUITY']     / (df['AMT_CREDIT'] + 1e-8)
    df['INCOME_PER_PERSON']     = df['AMT_INCOME_TOTAL'] / (df['CNT_FAM_MEMBERS'] + 1e-8)

    # ── Employment stability ────────────────────────────────────────────────
    df['EMPLOYED_TO_AGE_RATIO'] = df['DAYS_EMPLOYED'] / (df['DAYS_BIRTH'] + 1e-8)

    # ── External score aggregations (bureau credit scores) ──────────────────
    ext_cols = [c for c in ('EXT_SOURCE_1', 'EXT_SOURCE_2', 'EXT_SOURCE_3')
                if c in df.columns]
    if ext_cols:
        df['EXT_SCORE_MEAN'] = df[ext_cols].mean(axis=1)
        df['EXT_SCORE_MIN']  = df[ext_cols].min(axis=1)
        df['EXT_SCORE_STD']  = df[ext_cols].std(axis=1)
        df['EXT_SCORE_PROD'] = df[ext_cols].prod(axis=1)

    # ── Interactions: credit ratio × external score ─────────────────────────
    for col in ext_cols:
        df[f'{col}_X_CREDIT_RATIO'] = df[col] * df['CREDIT_INCOME_RATIO']

    # ── Document completeness ────────────────────────────────────────────────
    doc_cols = [c for c in df.columns if 'FLAG_DOCUMENT' in c]
    if doc_cols:
        df['DOC_SUBMITTED_COUNT'] = df[doc_cols].sum(axis=1)

    print(f'Domain features added. New shape: {df.shape}')
    return df


app_train_proc = engineer_domain_features(app_train_proc)
app_test_proc  = engineer_domain_features(app_test_proc)

In [ ]:
def assemble_master(app: pd.DataFrame,
                    bureau: pd.DataFrame,
                    prev: pd.DataFrame,
                    pos: pd.DataFrame,
                    inst: pd.DataFrame,
                    cc: pd.DataFrame) -> pd.DataFrame:
    '''Left-join all feature tables onto the application table via SK_ID_CURR.'''
    df = app.copy()
    secondary = [
        (bureau, 'bureau'),
        (prev,   'previous_application'),
        (pos,    'pos_cash'),
        (inst,   'installments'),
        (cc,     'credit_card'),
    ]
    for feat_df, name in secondary:
        before = df.shape[1]
        df = df.merge(feat_df, on='SK_ID_CURR', how='left')
        added = df.shape[1] - before
        print(f'  + {name:25s}  added {added:4d} features  →  total {df.shape[1]}')

    # Customers with no history in secondary tables get NaN from left join.
    # Impute with 0 — semantically: "no observed history" (not truly missing).
    num_cols = df.select_dtypes(include=[np.number]).columns
    df[num_cols] = df[num_cols].fillna(0)
    return df


print('Assembling train master table...')
train_master = assemble_master(
    app_train_proc, bureau_feat, prev_feat, pos_feat, inst_feat, cc_feat
)

print('\nAssembling test master table...')
test_master = assemble_master(
    app_test_proc, bureau_feat, prev_feat, pos_feat, inst_feat, cc_feat
)

print(f'\nFinal train shape : {train_master.shape}')
print(f'Final test shape  : {test_master.shape}')

---
## 6. Feature Selection

In [ ]:
def select_features(
    train: pd.DataFrame,
    target_col: str,
    corr_threshold: float = 0.95,
    importance_top_n: int = 400
) -> tuple:
    '''
    Three-stage feature selection:
      1. Remove zero-variance features
      2. Remove highly correlated features
      3. LightGBM importance ranking — keep top N
    '''
    exclude = {target_col, 'SK_ID_CURR'}
    feature_cols = [c for c in train.columns if c not in exclude]
    X = train[feature_cols]
    y = train[target_col]

    # Stage 1: Variance threshold
    vt = VarianceThreshold(threshold=0.0)
    vt.fit(X)
    stage1 = [feature_cols[i] for i, keep in enumerate(vt.get_support()) if keep]
    print(f'Stage 1 (zero variance)  : {len(feature_cols):4d} → {len(stage1):4d}  (removed {len(feature_cols)-len(stage1)})')

    # Stage 2: Correlation filter
    corr_matrix = train[stage1].corr().abs()
    upper = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))
    to_drop = [col for col in upper.columns if any(upper[col] > corr_threshold)]
    stage2 = [c for c in stage1 if c not in to_drop]
    print(f'Stage 2 (correlation>{corr_threshold}) : {len(stage1):4d} → {len(stage2):4d}  (removed {len(to_drop)})')

    # Stage 3: LightGBM importance
    probe_model = lgb.LGBMClassifier(
        n_estimators=300, learning_rate=0.05, num_leaves=31,
        scale_pos_weight=int((y == 0).sum() / (y == 1).sum()),
        random_state=SEED, n_jobs=-1, verbose=-1
    )
    probe_model.fit(X[stage2], y)
    imp_df = pd.DataFrame({
        'feature':    stage2,
        'importance': probe_model.feature_importances_
    }).sort_values('importance', ascending=False).reset_index(drop=True)

    stage3 = imp_df.head(importance_top_n)['feature'].tolist()
    print(f'Stage 3 (top {importance_top_n} by importance): {len(stage2):4d} → {len(stage3):4d}')

    # Plot top 30
    fig, ax = plt.subplots(figsize=(12, 9))
    top30 = imp_df.head(30)
    ax.barh(top30['feature'][::-1], top30['importance'][::-1],
            color=plt.cm.viridis(np.linspace(0.2, 0.9, 30))[::-1])
    ax.set_title('Top 30 Features by LightGBM Importance', fontsize=13, fontweight='bold')
    ax.set_xlabel('Importance Score')
    plt.tight_layout()
    plt.savefig(f'{OUT_DIR}/05_feature_importance.png', dpi=150, bbox_inches='tight')
    plt.show()

    return stage3, imp_df


selected_features, importance_df = select_features(train_master, TARGET)
print(f'\nFinal feature count: {len(selected_features)}')

# ── Populate ground truth ──────────────────────────────────────────────────────
ground_truth['top_20_features']    = importance_df.head(20)[['feature', 'importance']].to_dict('records')
ground_truth['n_selected_features'] = len(selected_features)
ground_truth['ext_source_ranks']   = {
    col: int(importance_df[importance_df['feature'] == col].index[0]) + 1
    for col in ['EXT_SOURCE_1', 'EXT_SOURCE_2', 'EXT_SOURCE_3']
    if col in importance_df['feature'].values
}

---
## 7. Modeling

In [ ]:
# ── Prepare matrices ───────────────────────────────────────────────────────────
X_train = train_master[selected_features].values
y_train = train_master[TARGET].values
X_test  = test_master[selected_features].values

pos_weight = int((y_train == 0).sum() / (y_train == 1).sum())
print(f'scale_pos_weight: {pos_weight}  (used to compensate class imbalance)')
print(f'Training samples : {len(X_train):,}')
print(f'Features         : {X_train.shape[1]}')

In [ ]:
def train_lgbm_cv(
    X: np.ndarray, y: np.ndarray, X_test: np.ndarray,
    feature_names: list, n_folds: int = N_FOLDS
) -> tuple:
    '''LightGBM with StratifiedKFold cross-validation.'''
    oof_preds  = np.zeros(len(X))
    test_preds = np.zeros(len(X_test))
    cv_scores  = []
    last_model = None

    lgb_params = {
        'objective':        'binary',
        'metric':           'auc',
        'learning_rate':    0.05,
        'num_leaves':       63,
        'max_depth':        -1,
        'min_child_samples': 20,
        'subsample':        0.8,
        'colsample_bytree': 0.8,
        'reg_alpha':        0.1,
        'reg_lambda':       0.1,
        'scale_pos_weight': int((y == 0).sum() / (y == 1).sum()),
        'random_state':     SEED,
        'n_jobs':           -1,
        'verbose':          -1,
    }

    skf = StratifiedKFold(n_splits=n_folds, shuffle=True, random_state=SEED)

    for fold, (tr_idx, val_idx) in enumerate(skf.split(X, y), 1):
        X_tr, X_val = X[tr_idx], X[val_idx]
        y_tr, y_val = y[tr_idx], y[val_idx]

        dtrain = lgb.Dataset(X_tr, label=y_tr,  feature_name=feature_names)
        dval   = lgb.Dataset(X_val, label=y_val, reference=dtrain)

        model = lgb.train(
            lgb_params, dtrain,
            num_boost_round=3000,
            valid_sets=[dval],
            callbacks=[
                lgb.early_stopping(stopping_rounds=100, verbose=False),
                lgb.log_evaluation(period=-1)
            ]
        )

        oof_preds[val_idx] = model.predict(X_val, num_iteration=model.best_iteration)
        test_preds += model.predict(X_test, num_iteration=model.best_iteration) / n_folds
        fold_auc = roc_auc_score(y_val, oof_preds[val_idx])
        cv_scores.append(fold_auc)
        print(f'  Fold {fold}/{n_folds}  AUC: {fold_auc:.5f}  best_iter: {model.best_iteration}')
        last_model = model

    oof_auc = roc_auc_score(y, oof_preds)
    print(f'\n  OOF AUC  : {oof_auc:.5f}')
    print(f'  CV Mean  : {np.mean(cv_scores):.5f} ± {np.std(cv_scores):.5f}')
    return oof_preds, test_preds, cv_scores, oof_auc, last_model


print(f'Training LightGBM with {N_FOLDS}-fold CV...')
lgb_oof, lgb_test, lgb_cv, lgb_auc, lgb_model = train_lgbm_cv(
    X_train, y_train, X_test, selected_features
)

# ── Populate ground truth ──────────────────────────────────────────────────────
ground_truth['oof_auc']  = float(lgb_auc)
ground_truth['cv_mean']  = float(np.mean(lgb_cv))
ground_truth['cv_std']   = float(np.std(lgb_cv))

In [ ]:
# ── XGBoost (for ensemble / comparison) ───────────────────────────────────────
from xgboost import XGBClassifier

xgb_oof  = np.zeros(len(X_train))
xgb_test = np.zeros(len(X_test))
xgb_scores = []

xgb_params = dict(
    n_estimators=2000, learning_rate=0.05, max_depth=6,
    subsample=0.8, colsample_bytree=0.8,
    scale_pos_weight=pos_weight,
    eval_metric='auc', use_label_encoder=False,
    random_state=SEED, n_jobs=-1, verbosity=0
)

skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)

for fold, (tr_idx, val_idx) in enumerate(skf.split(X_train, y_train), 1):
    xgb_model = XGBClassifier(**xgb_params)
    xgb_model.fit(
        X_train[tr_idx], y_train[tr_idx],
        eval_set=[(X_train[val_idx], y_train[val_idx])],
        early_stopping_rounds=100, verbose=False
    )
    xgb_oof[val_idx] = xgb_model.predict_proba(X_train[val_idx])[:, 1]
    xgb_test += xgb_model.predict_proba(X_test)[:, 1] / N_FOLDS
    fold_auc = roc_auc_score(y_train[val_idx], xgb_oof[val_idx])
    xgb_scores.append(fold_auc)
    print(f'  XGB Fold {fold}/{N_FOLDS}  AUC: {fold_auc:.5f}')

xgb_auc = roc_auc_score(y_train, xgb_oof)
print(f'\n  XGB OOF AUC: {xgb_auc:.5f}')
print(f'  XGB CV Mean: {np.mean(xgb_scores):.5f} ± {np.std(xgb_scores):.5f}')

In [ ]:
from sklearn.preprocessing import RobustScaler

lr_oof    = np.zeros(len(X_train))
lr_scores = []
scaler    = RobustScaler()

for fold, (tr_idx, val_idx) in enumerate(skf.split(X_train, y_train), 1):
    X_tr_sc  = scaler.fit_transform(X_train[tr_idx])
    X_val_sc = scaler.transform(X_train[val_idx])
    lr = LogisticRegression(
        C=0.01, max_iter=1000, class_weight='balanced',
        solver='saga', random_state=SEED, n_jobs=-1
    )
    lr.fit(X_tr_sc, y_train[tr_idx])
    lr_oof[val_idx] = lr.predict_proba(X_val_sc)[:, 1]
    lr_scores.append(roc_auc_score(y_train[val_idx], lr_oof[val_idx]))
    print(f'  LR Fold {fold}/{N_FOLDS}  AUC: {lr_scores[-1]:.5f}')

lr_auc = roc_auc_score(y_train, lr_oof)
print(f'\n  LR OOF AUC: {lr_auc:.5f}')

print('\n=== Model Comparison ===')
for name, auc in [('Logistic Regression', lr_auc), ('XGBoost', xgb_auc), ('LightGBM', lgb_auc)]:
    print(f'  {name:22s}: AUC = {auc:.5f}')

# ── Populate ground truth ──────────────────────────────────────────────────────
ground_truth['xgb_auc'] = float(xgb_auc)
ground_truth['lr_auc']  = float(lr_auc)

---
## 8. Evaluation & Business KPIs

In [ ]:
# ── ROC and Precision-Recall curves ───────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

models = [
    ('LightGBM', lgb_oof, lgb_auc),
    ('XGBoost',  xgb_oof, xgb_auc),
    ('Log. Regression', lr_oof, lr_auc)
]
colors_m = ['#E74C3C', '#3498DB', '#2ECC71']

# ROC
for (name, preds, auc_val), color in zip(models, colors_m):
    fpr, tpr, _ = roc_curve(y_train, preds)
    axes[0].plot(fpr, tpr, color=color, linewidth=2,
                 label=f'{name} (AUC={auc_val:.4f})')
axes[0].plot([0, 1], [0, 1], 'k--', alpha=0.4, label='Random')
axes[0].set_xlabel('False Positive Rate', fontsize=11)
axes[0].set_ylabel('True Positive Rate', fontsize=11)
axes[0].set_title('ROC Curve', fontsize=13, fontweight='bold')
axes[0].legend(fontsize=9)
axes[0].grid(True, alpha=0.3)

# Precision-Recall
for (name, preds, _), color in zip(models, colors_m):
    prec, rec, _ = precision_recall_curve(y_train, preds)
    ap = average_precision_score(y_train, preds)
    axes[1].plot(rec, prec, color=color, linewidth=2,
                 label=f'{name} (AP={ap:.4f})')
axes[1].axhline(y=y_train.mean(), color='black', linestyle='--',
                alpha=0.5, label=f'Baseline ({y_train.mean():.3f})')
axes[1].set_xlabel('Recall', fontsize=11)
axes[1].set_ylabel('Precision', fontsize=11)
axes[1].set_title('Precision-Recall Curve', fontsize=13, fontweight='bold')
axes[1].legend(fontsize=9)
axes[1].grid(True, alpha=0.3)

plt.suptitle('Model Evaluation', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(f'{OUT_DIR}/06_roc_pr_curves.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
def compute_ks_statistic(y_true: np.ndarray, y_score: np.ndarray,
                          model_name: str = 'Model') -> float:
    '''
    Kolmogorov-Smirnov (KS) statistic — primary metric in credit risk industry.
    Benchmarks: <20% Poor | 20-40% Acceptable | 40-50% Good | 50-60% Very Good | >60% Excellent
    '''
    bads  = y_score[y_true == 1]
    goods = y_score[y_true == 0]
    ks_stat, _ = stats.ks_2samp(bads, goods)

    thresholds = np.linspace(0, 1, 1000)
    tpr_arr = np.array([np.mean(bads >= t) for t in thresholds])
    fpr_arr = np.array([np.mean(goods >= t) for t in thresholds])
    ks_curve = tpr_arr - fpr_arr
    ks_idx   = np.argmax(ks_curve)

    fig, ax = plt.subplots(figsize=(10, 6))
    ax.plot(thresholds, tpr_arr, 'r-',  linewidth=2, label='Default CDF')
    ax.plot(thresholds, fpr_arr, 'g-',  linewidth=2, label='Non-Default CDF')
    ax.plot(thresholds, ks_curve, 'b--', linewidth=2, label=f'KS = {ks_stat:.4f}')
    ax.axvline(x=thresholds[ks_idx], color='purple', linestyle=':',
               alpha=0.9, label=f'Opt. threshold ≈ {thresholds[ks_idx]:.3f}')
    ax.fill_between(thresholds, fpr_arr, tpr_arr, alpha=0.1, color='blue')
    ax.set_xlabel('Score Threshold', fontsize=11)
    ax.set_ylabel('Cumulative Rate',  fontsize=11)
    ax.set_title(f'{model_name} — KS = {ks_stat*100:.1f}%', fontsize=13, fontweight='bold')
    ax.legend(fontsize=10)
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig(f'{OUT_DIR}/07_ks_statistic.png', dpi=150, bbox_inches='tight')
    plt.show()
    return ks_stat


ks_lgbm = compute_ks_statistic(y_train, lgb_oof, 'LightGBM')
gini    = 2 * lgb_auc - 1

print(f'\n=== Credit Risk Industry KPIs ===')
print(f'  AUC-ROC : {lgb_auc:.5f}')
print(f'  Gini    : {gini:.5f}   (= 2 × AUC − 1)')
print(f'  KS      : {ks_lgbm:.5f}  ({ks_lgbm*100:.1f}%)')

# ── Populate ground truth ──────────────────────────────────────────────────────
ground_truth['ks_stat'] = float(ks_lgbm)
ground_truth['gini']    = float(gini)

In [ ]:
def compute_business_metrics(
    y_true: np.ndarray, y_score: np.ndarray,
    cost_fn: float = 5000, cost_fp: float = 500
) -> tuple:
    '''Cost-sensitive threshold analysis. cost_fn = approving a defaulter, cost_fp = rejecting a good customer.'''
    thresholds = np.arange(0.05, 0.95, 0.005)
    rows = []
    for t in thresholds:
        y_pred = (y_score >= t).astype(int)
        tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()
        rows.append({
            'threshold':     t,
            'approval_rate': (tn + fp) / len(y_true),
            'capture_rate':  tp / (tp + fn + 1e-8),
            'precision':     tp / (tp + fp + 1e-8),
            'total_cost':    fp * cost_fp + fn * cost_fn,
            'tp': tp, 'fp': fp, 'tn': tn, 'fn': fn
        })
    df = pd.DataFrame(rows)
    opt_idx = df['total_cost'].idxmin()
    opt_t   = df.loc[opt_idx, 'threshold']

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    axes[0].plot(df['threshold'], df['total_cost'], 'b-', linewidth=2)
    axes[0].axvline(opt_t, color='red', linestyle='--', label=f'Optimal = {opt_t:.3f}')
    axes[0].set_xlabel('Decision Threshold')
    axes[0].set_ylabel('Total Business Cost ($)')
    axes[0].set_title(f'Business Cost vs Threshold (FN=${cost_fn:,}, FP=${cost_fp:,})', fontweight='bold')
    axes[0].legend()
    axes[0].grid(True, alpha=0.3)

    axes[1].plot(df['threshold'], df['approval_rate'] * 100, 'g-', linewidth=2, label='Approval Rate (%)')
    axes[1].plot(df['threshold'], df['capture_rate']  * 100, 'r-', linewidth=2, label='Default Capture Rate (%)')
    axes[1].axvline(opt_t, color='purple', linestyle='--', alpha=0.7)
    axes[1].set_xlabel('Decision Threshold')
    axes[1].set_ylabel('Rate (%)')
    axes[1].set_title('Approval Rate vs Default Capture Rate', fontweight='bold')
    axes[1].legend()
    axes[1].grid(True, alpha=0.3)

    plt.suptitle('Cost-Sensitive Threshold Analysis', fontsize=13, fontweight='bold')
    plt.tight_layout()
    plt.savefig(f'{OUT_DIR}/08_business_metrics.png', dpi=150, bbox_inches='tight')
    plt.show()

    print(f'Optimal threshold: {opt_t:.3f}')
    print(df.loc[opt_idx][['approval_rate', 'capture_rate', 'precision', 'total_cost']])
    return df, opt_t


biz_df, opt_threshold = compute_business_metrics(y_train, lgb_oof)

# ── Populate ground truth ──────────────────────────────────────────────────────
ground_truth['optimal_threshold']      = float(opt_threshold)
ground_truth['approval_rate_optimal']  = float(biz_df.loc[biz_df['threshold'].sub(opt_threshold).abs().idxmin(), 'approval_rate'])

# Confusion matrix at default threshold=0.5 (to show Gemini why it's wrong)
_cm_05 = confusion_matrix(y_train, (lgb_oof >= 0.5).astype(int))
ground_truth['cm_threshold_05']        = _cm_05.tolist()
ground_truth['approval_rate_05']       = float((lgb_oof < 0.5).mean())

In [ ]:
# ── SHAP Feature Importance & Interpretability ────────────────────────────────
sample_idx = np.random.choice(len(X_train), size=min(2000, len(X_train)), replace=False)
X_sample   = X_train[sample_idx]

explainer   = shap.TreeExplainer(lgb_model)
shap_values = explainer.shap_values(X_sample)

if isinstance(shap_values, list):
    sv = shap_values[1]   # class=1 (default)
else:
    sv = shap_values

# SHAP Summary plot
plt.figure(figsize=(12, 10))
shap.summary_plot(
    sv, X_sample,
    feature_names=selected_features,
    max_display=20, show=False
)
plt.title('SHAP Summary — Top 20 Features', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(f'{OUT_DIR}/09_shap_summary.png', dpi=150, bbox_inches='tight')
plt.show()

# Mean |SHAP| bar chart
mean_shap = np.abs(sv).mean(axis=0)
shap_df = pd.DataFrame({'feature': selected_features, 'mean_shap': mean_shap})
shap_df = shap_df.sort_values('mean_shap', ascending=False).head(20)

fig, ax = plt.subplots(figsize=(11, 7))
ax.barh(shap_df['feature'][::-1], shap_df['mean_shap'][::-1],
        color='#3498DB', edgecolor='white')
ax.set_xlabel('Mean |SHAP Value|', fontsize=11)
ax.set_title('SHAP Global Feature Importance (Top 20)', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(f'{OUT_DIR}/10_shap_bar.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
decile_df = pd.DataFrame({'score': lgb_oof, 'target': y_train})
decile_df['decile'] = pd.qcut(decile_df['score'], q=10, labels=False, duplicates='drop') + 1

decile_summary = decile_df.groupby('decile').agg(
    n_customers    = ('target', 'count'),
    n_defaults     = ('target', 'sum'),
    default_rate   = ('target', 'mean'),
    avg_score      = ('score',  'mean')
).reset_index()

decile_summary['cum_default_capture'] = (
    decile_summary['n_defaults'][::-1].cumsum()[::-1] /
    decile_summary['n_defaults'].sum()
)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].bar(decile_summary['decile'], decile_summary['default_rate'] * 100,
            color=plt.cm.RdYlGn_r(np.linspace(0.1, 0.9, 10)), edgecolor='white')
axes[0].axhline(y_train.mean() * 100, color='black', linestyle='--',
                alpha=0.6, label=f'Overall rate ({y_train.mean():.1%})')
axes[0].set_xlabel('Score Decile (1=lowest, 10=highest)')
axes[0].set_ylabel('Default Rate (%)')
axes[0].set_title('Default Rate by Score Decile', fontweight='bold')
axes[0].legend()
axes[0].grid(True, alpha=0.3, axis='y')

axes[1].plot(decile_summary['decile'][::-1],
             decile_summary['cum_default_capture'][::-1] * 100, 'b-o', linewidth=2)
axes[1].set_xlabel('Top N Deciles Selected')
axes[1].set_ylabel('% of Defaults Captured')
axes[1].set_title('Cumulative Default Capture Rate', fontweight='bold')
axes[1].grid(True, alpha=0.3)

plt.suptitle('Scorecard KPIs', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(f'{OUT_DIR}/11_scorecard_kpis.png', dpi=150, bbox_inches='tight')
plt.show()

print('Default Rate by Decile:')
print(decile_summary.to_string(index=False))

# ── Populate ground truth ──────────────────────────────────────────────────────
ground_truth['default_rate_by_decile'] = decile_summary[['decile', 'n_customers', 'default_rate']].to_dict('records')
ground_truth['top_decile_default_rate'] = float(decile_summary[decile_summary['decile'] == decile_summary['decile'].max()]['default_rate'].iloc[0])
print(f'\nground_truth dict now has {len(ground_truth)} keys — pipeline complete.')

---
## 9. Headroom Dataset Export

In [ ]:
# ── Export headroom dataset ────────────────────────────────────────────────────
headroom_df = logger.to_dataframe()

print(f'Total headroom tasks logged: {len(headroom_df)}')
print('\nSummary by stage:')
print(headroom_df.groupby('stage')['task_id'].count().to_string())

print('\nSummary by failure category:')
print(headroom_df.groupby('failure_category')['task_id'].count().to_string())

print('\nSummary by difficulty:')
print(headroom_df.groupby('difficulty')['task_id'].count().to_string())

# Export
logger.export('/content/outputs/headroom_dataset')

print('\nFiles exported:')
print('  /content/outputs/headroom_dataset.csv')
print('  /content/outputs/headroom_dataset.json')

---
## Phase 2: Gemini Evaluation — Headroom Task Creation & Testing

**Design:** Gemini is given **real pipeline outputs** as context (actual numbers, actual feature lists, actual confusion matrices) and asked to make the same decisions a data scientist would.

**Ground truth** comes from actually running the pipeline above — LightGBM tells us the real answers.

**Headroom** = tasks where Gemini's response diverges from the pipeline-verified ground truth.

---
## Phase 2a: Gemini End-to-End Analysis (Agentic Loop)

Gemini receives the same DS challenge as Claude: build a full ML pipeline and produce predictions.
Both models use the **same agentic loop via OpenRouter** — they call `execute_python`, we run
the code in the Colab kernel where all DataFrames are already loaded, and loop until done.

**1 API key**: `OPENROUTER_API_KEY` handles both models.


In [ ]:

# ── Shared agentic infrastructure (Gemini + Claude, OpenRouter only) ──────────
%%capture
!pip install openai -q

import json, io, contextlib, traceback, textwrap, time
from openai import OpenAI

# ── OpenRouter config — set your key here ─────────────────────────────────────
OPENROUTER_API_KEY = 'YOUR_OPENROUTER_KEY_HERE'   # ← replace with your key

GEMINI_MODEL = 'google/gemini-2.0-pro-exp-02-05'  # update slug as needed
CLAUDE_MODEL = 'anthropic/claude-opus-4-6'

client = OpenAI(
    base_url = 'https://openrouter.ai/api/v1',
    api_key  = OPENROUTER_API_KEY
)

# ── Tool definition for both models ───────────────────────────────────────────
AGENT_TOOL = [{
    "type": "function",
    "function": {
        "name": "execute_python",
        "description": (
            "Execute Python code in the Colab kernel. "
            "All DataFrames (app_train, app_test, bureau, bureau_bal, prev_app, "
            "pos_cash, installments, credit_card) and all pipeline variables persist "
            "between calls. Use print() to show results."
        ),
        "parameters": {
            "type": "object",
            "properties": {
                "code": {"type": "string", "description": "Python code to execute"}
            },
            "required": ["code"]
        }
    }
}]


def safe_exec(code: str) -> str:
    '''Execute code in the shared Colab kernel globals; return stdout/stderr.'''
    buf = io.StringIO()
    try:
        with contextlib.redirect_stdout(buf):
            exec(textwrap.dedent(code), globals())
        output = buf.getvalue()
        return output if output.strip() else '(executed — no stdout output)'
    except Exception:
        return traceback.format_exc()[-2000:]


def run_agent(model: str, system_prompt: str, user_prompt: str,
              max_turns: int = 25, verbose: bool = True) -> tuple:
    '''
    Agentic loop for any OpenRouter model.
    Model writes code → we execute in Colab → return output → loop until done.
    Returns (final_response, list_of_code_blocks_executed).
    '''
    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user",   "content": user_prompt}
    ]
    all_code = []

    for turn in range(max_turns):
        resp = client.chat.completions.create(
            model       = model,
            tools       = AGENT_TOOL,
            messages    = messages,
            max_tokens  = 4096,
            temperature = 0.0
        )
        msg = resp.choices[0].message

        asst = {"role": "assistant", "content": msg.content or ""}
        if msg.tool_calls:
            asst["tool_calls"] = msg.tool_calls
        messages.append(asst)

        if resp.choices[0].finish_reason != "tool_calls" or not msg.tool_calls:
            if verbose:
                print(f'  → {model.split("/")[-1]} done after {turn+1} turns | '
                      f'{len(all_code)} code blocks executed.')
            break

        for tc in msg.tool_calls:
            code   = json.loads(tc.function.arguments)["code"]
            all_code.append(code)
            output = safe_exec(code)
            if verbose:
                print(f'  [t{turn+1}] {len(code):4d}c → {len(output):4d}c output')
            messages.append({
                "role": "tool",
                "tool_call_id": tc.id,
                "content": output[:3000]
            })

    return resp, all_code


print(f'OpenRouter client ready.')
print(f'  Gemini : {GEMINI_MODEL}')
print(f'  Claude : {CLAUDE_MODEL}')
print(f'run_agent() and safe_exec() defined.')


In [ ]:

# ── Gemini end-to-end agentic DS run ─────────────────────────────────────────
DS_SYSTEM = (
    'You are a world-class data scientist. You have access to a Python execution '
    'environment where the Home Credit Default Risk dataset is already loaded:\n'
    '  app_train, app_test, bureau, bureau_bal, prev_app, pos_cash, '
    'installments, credit_card\n'
    'All standard libraries (pandas, numpy, sklearn, lightgbm, scipy) are available.\n'
    'Work systematically: explore → engineer features → train model → evaluate → predict.\n'
    'Save final test predictions as a variable named after your model (e.g. gemini_test_preds).\n'
    'Think step by step and verify each step with print statements.'
)

GEMINI_E2E_PROMPT = """\
Build a complete end-to-end ML pipeline for the Home Credit Default Risk dataset.

Goals:
1. EDA — identify the most critical data quality issue
2. Data Preprocessing - Do all the data preprocessing you need to do
3. Feature engineering — aggregate auxiliary tables (bureau, previous_application,
   installments, POS_CASH, credit_card) into per-applicant features
4. Model — train a LightGBM classifier with 5-fold StratifiedKFold CV;
   handle class imbalance; report hyperparameter choices
5. Evaluate — OOF AUC, KS statistic, Gini = 2*AUC-1, optimal classification threshold
6. Predict — generate test set predictions; store as `gemini_test_preds`

Use execute_python to run code iteratively. Explore first, then build, then evaluate.
"""

print('Launching Gemini agentic data scientist via OpenRouter...\n')
gemini_e2e_resp, gemini_e2e_code = run_agent(
    model         = GEMINI_MODEL,
    system_prompt = DS_SYSTEM,
    user_prompt   = GEMINI_E2E_PROMPT,
    max_turns     = 30,
    verbose       = True
)
print(f'\nGemini used {len(gemini_e2e_code)} code executions.')


In [ ]:

# ── Display Gemini's E2E analysis summary ────────────────────────────────────
# Save Gemini's submission if it produced predictions
if 'gemini_test_preds' in globals():
    import pandas as pd
    submission_gemini = pd.DataFrame({
        'SK_ID_CURR': app_test['SK_ID_CURR'].astype(int),
        'TARGET':     gemini_test_preds
    })
    submission_gemini.to_csv('/content/outputs/submission_gemini.csv', index=False)
    print(f'Gemini submission saved: {len(submission_gemini)} rows')
    print(f'Prediction range: [{gemini_test_preds.min():.4f}, {gemini_test_preds.max():.4f}]')
else:
    print('WARNING: gemini_test_preds not found — Gemini may not have finished the pipeline.')

# Print final text from Gemini's last message
print('\n' + '='*70)
print('GEMINI — FINAL ANALYSIS SUMMARY')
print('='*70)
final_text = gemini_e2e_resp.choices[0].message.content or '(no final text)'
print(final_text[:3000])
print(f'\n[{len(gemini_e2e_code)} code blocks executed in total]')


In [ ]:

# ── Extract Gemini's reported metrics from globals (set by its code) ──────────
gemini_metrics = {
    'oof_auc':          globals().get('gemini_oof_auc'),
    'ks_stat':          globals().get('gemini_ks'),
    'gini':             globals().get('gemini_gini'),
    'optimal_threshold':globals().get('gemini_threshold'),
    'n_code_blocks':    len(gemini_e2e_code),
}

print('Gemini self-reported metrics (if it assigned these variables):')
for k, v in gemini_metrics.items():
    print(f'  {k:22s}: {v}')


In [ ]:

# ── Compare Gemini E2E to LightGBM ground truth ───────────────────────────────
print('=' * 60)
print('E2E COMPARISON: LightGBM vs Gemini')
print('=' * 60)

gt_auc = ground_truth.get('oof_auc')
gt_ks  = ground_truth.get('ks_stat')
gt_gi  = ground_truth.get('gini')
gt_thr = ground_truth.get('optimal_threshold')

rows = [
    ('LightGBM (baseline)', gt_auc,  gt_ks,  gt_gi,  gt_thr),
    ('Gemini (agentic)',    gemini_metrics['oof_auc'], gemini_metrics['ks_stat'],
     gemini_metrics['gini'], gemini_metrics['optimal_threshold']),
]
print(f"{'Model':<24} {'OOF AUC':>8} {'KS':>8} {'Gini':>8} {'Threshold':>10}")
print('-' * 60)
for name, auc, ks, gini, thr in rows:
    print(f'{name:<24} '
          f'{str(round(auc,4)) if auc else "N/A":>8} '
          f'{str(round(ks,4))  if ks  else "N/A":>8} '
          f'{str(round(gini,4))if gini else "N/A":>8} '
          f'{str(round(thr,3)) if thr  else "N/A":>10}')

# Keyword-based checkpoint scoring on Gemini's final text
gemini_final = gemini_e2e_resp.choices[0].message.content or ''
CHECKPOINTS_GEMINI = [
    {'id':'CP-G1','stage':'EDA',         'keywords':['365243','anomaly','days_employed']},
    {'id':'CP-G2','stage':'Feat Eng',    'keywords':['bureau','aggregat','previous']},
    {'id':'CP-G3','stage':'Modeling',    'keywords':['lightgbm','stratified','fold']},
    {'id':'CP-G4','stage':'Evaluation',  'keywords':['auc','ks','gini','roc']},
    {'id':'CP-G5','stage':'Threshold',   'keywords':['threshold','cost','0.5']},
    {'id':'CP-G6','stage':'Regulatory',  'keywords':['ecoa','fair','shap','explainab']},
]
gemini_results = []
for cp in CHECKPOINTS_GEMINI:
    passed = any(kw in gemini_final.lower() for kw in cp['keywords'])
    gemini_results.append({**cp, 'passed': passed})
    print(f'{"✓ PASS" if passed else "✗ FAIL"} | {cp["id"]} {cp["stage"]}')

gemini_pass_rate = sum(r['passed'] for r in gemini_results) / len(gemini_results)
print(f'\nGemini E2E analysis pass rate: {gemini_pass_rate:.0%}')


---
## Phase 2b: Claude Opus 4.6 — End-to-End Implementation (Agentic)

Claude Opus 4.6 receives the **same DS challenge** as Gemini via the **same `run_agent()` loop**
using OpenRouter. Same system prompt, same tool definition, same Colab kernel — fully fair comparison.

Both models: write code → `safe_exec()` runs it in Colab → model sees output → iterates until done.
Only **1 API key needed**: `OPENROUTER_API_KEY` (set in cell 1kp708fammd above).


In [ ]:

# ── Claude Opus 4.6 end-to-end agentic run ────────────────────────────────────
# Uses the same run_agent() + safe_exec() defined in Phase 2a (cell 1kp708fammd)
# Same DS_SYSTEM prompt as Gemini for a fair comparison

CLAUDE_E2E_PROMPT = """\
Build a complete end-to-end ML pipeline for the Home Credit Default Risk dataset.

Goals:
1. EDA — identify the most critical data quality issue (check DAYS_EMPLOYED values carefully)
2. Data Preprocessing - Do all the data preprocessing you need to do
3. Feature engineering — aggregate auxiliary tables (bureau, previous_application,
   installments, POS_CASH, credit_card) into per-applicant features
4. Model — train a LightGBM classifier with 5-fold StratifiedKFold CV;
   handle class imbalance explicitly; document hyperparameter choices
5. Evaluate — OOF AUC, KS statistic, Gini = 2*AUC-1, optimal classification threshold
6. Predict — generate test set predictions; store as `claude_test_preds`

Use execute_python to run code iteratively. Explore first, then build, then evaluate.
Print all key metrics clearly.
"""

print('Launching Claude Opus 4.6 agentic data scientist via OpenRouter...\n')
claude_e2e_resp, claude_e2e_code = run_agent(
    model         = CLAUDE_MODEL,
    system_prompt = DS_SYSTEM,          # shared with Gemini — same system prompt
    user_prompt   = CLAUDE_E2E_PROMPT,
    max_turns     = 30,
    verbose       = True
)
print(f'\nClaude used {len(claude_e2e_code)} code executions.')


In [ ]:

# ── Save Claude's submission + extract metrics ─────────────────────────────────
if 'claude_test_preds' in globals():
    submission_claude = pd.DataFrame({
        'SK_ID_CURR': app_test['SK_ID_CURR'].astype(int),
        'TARGET':     claude_test_preds
    })
    submission_claude.to_csv('/content/outputs/submission_claude.csv', index=False)
    print(f'Claude submission saved: {len(submission_claude)} rows')
    print(f'Prediction range: [{claude_test_preds.min():.4f}, {claude_test_preds.max():.4f}]')
else:
    print('WARNING: claude_test_preds not found — Claude may not have finished the pipeline.')

claude_final = claude_e2e_resp.choices[0].message.content or ''
claude_metrics = {
    'oof_auc':           globals().get('claude_oof_auc'),
    'ks_stat':           globals().get('claude_ks'),
    'gini':              globals().get('claude_gini'),
    'optimal_threshold': globals().get('claude_threshold'),
    'n_code_blocks':     len(claude_e2e_code),
}

print('\n' + '='*70)
print('CLAUDE OPUS 4.6 — FINAL ANALYSIS SUMMARY')
print('='*70)
print(claude_final[:3000])


In [ ]:

# ── Compare Claude E2E to LightGBM ground truth ───────────────────────────────
print('=' * 60)
print('E2E COMPARISON: LightGBM vs Claude Opus 4.6')
print('=' * 60)

rows = [
    ('LightGBM (baseline)', ground_truth.get('oof_auc'),  ground_truth.get('ks_stat'),
     ground_truth.get('gini'), ground_truth.get('optimal_threshold')),
    ('Claude Opus 4.6',     claude_metrics['oof_auc'], claude_metrics['ks_stat'],
     claude_metrics['gini'], claude_metrics['optimal_threshold']),
]
print(f"{'Model':<24} {'OOF AUC':>8} {'KS':>8} {'Gini':>8} {'Threshold':>10}")
print('-' * 60)
for name, auc, ks, gini, thr in rows:
    print(f'{name:<24} '
          f'{str(round(auc,4))  if auc  else "N/A":>8} '
          f'{str(round(ks,4))   if ks   else "N/A":>8} '
          f'{str(round(gini,4)) if gini else "N/A":>8} '
          f'{str(round(thr,3))  if thr  else "N/A":>10}')

# Checkpoint scoring on Claude's final analysis text
CHECKPOINTS_CLAUDE = [
    {'id': 'CP-C1', 'stage': 'EDA',        'keywords': ['365243', 'anomaly', 'days_employed']},
    {'id': 'CP-C2', 'stage': 'Feat Eng',   'keywords': ['bureau', 'aggregat', 'previous']},
    {'id': 'CP-C3', 'stage': 'Modeling',   'keywords': ['lightgbm', 'stratified', 'fold']},
    {'id': 'CP-C4', 'stage': 'Evaluation', 'keywords': ['auc', 'ks', 'gini', 'roc']},
    {'id': 'CP-C5', 'stage': 'Threshold',  'keywords': ['threshold', 'cost', '0.5']},
    {'id': 'CP-C6', 'stage': 'Regulatory', 'keywords': ['ecoa', 'fair', 'shap', 'explainab']},
]
claude_results = []
claude_final_lower = claude_final.lower()
for cp in CHECKPOINTS_CLAUDE:
    passed = any(kw in claude_final_lower for kw in cp['keywords'])
    claude_results.append({**cp, 'passed': passed})
    print(f'{"✓ PASS" if passed else "✗ FAIL"} | {cp["id"]} {cp["stage"]}')

claude_pass_rate = sum(r['passed'] for r in claude_results) / len(claude_results)
print(f'\nClaude E2E analysis pass rate: {claude_pass_rate:.0%}')
print(f'Gemini E2E analysis pass rate: {gemini_pass_rate:.0%}')
print(f'Headroom gap: Claude − Gemini = {(claude_pass_rate - gemini_pass_rate)*100:+.0f} pp')


In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# Create all 10 headroom tasks using REAL pipeline outputs as context
# ══════════════════════════════════════════════════════════════════════════════

pos_rate      = float(y_train.mean())
neg_pos_ratio = int((y_train == 0).sum() / (y_train == 1).sum())

# ── HT-001: DAYS_EMPLOYED anomaly ─────────────────────────────────────────────
logger.log(
    stage='Preprocessing',
    prompt='The feature DAYS_EMPLOYED contains a large cluster of values equal to 365243. What does this represent and what is the correct preprocessing strategy?',
    data_context=(
        f'Feature: DAYS_EMPLOYED (days before application person started current employment)\n'
        f'Typical values are negative (e.g. -711, -1134 = days in the past)\n\n'
        f'Value distribution:\n'
        f'  Non-365243 values: min=-17912, max=0, median={ground_truth["days_employed_non_anom_median"]:.0f}\n'
        f'  Value 365243: appears {ground_truth["days_employed_anom_count"]:,} times '
        f'({ground_truth["days_employed_anom_pct"]:.1%} of all applicants)\n\n'
        f'Note: 365243 days ≈ 1000 years — clearly not a real employment duration.'
    ),
    ground_truth=(
        f'365243 is a domain-specific placeholder for RETIRED/UNEMPLOYED applicants (no current employment). '
        f'Correct strategy: (1) Create binary flag DAYS_EMPLOYED_ANOM=1 for these {ground_truth["days_employed_anom_count"]:,} rows. '
        f'(2) Replace 365243 with NaN. (3) Impute with median of valid values. '
        f'The flag is a top predictive feature confirmed by LightGBM importance. '
        f'Wrong: treating as outlier/capping, imputing without flag, or dropping rows.'
    ),
    failure_category='domain_knowledge',
    difficulty='hard'
)

# ── HT-002: Missing value strategy ────────────────────────────────────────────
missing_str = '\n'.join([
    f'  {r["feature"]}: {r["missing_pct"]:.1f}%'
    for r in ground_truth.get('missing_top10', [])[:8]
])
logger.log(
    stage='Preprocessing',
    prompt='Given the missing value rates below, describe your complete preprocessing strategy for each group of features.',
    data_context=(
        f'Top features by missing value percentage (application_train.csv):\n{missing_str}\n\n'
        f'Total features with >50% missing: {ground_truth["missing_over50_count"]}\n'
        f'Total features with >20% missing: {ground_truth["missing_over20_count"]}'
    ),
    ground_truth=(
        'Strategy by group: '
        '>50% missing: create binary missing indicator + impute with median (or consider dropping). '
        '20-50% missing: impute with median/mode + create missing indicator flag. '
        '<20% missing: impute with median/mode only. '
        'XNA values = explicit missing code in this dataset — replace with NaN before imputation. '
        'Key insight: missingness itself is informative — always create indicators for high-missing features. '
        'Gemini failure: suggests mean imputation without creating missing indicators.'
    ),
    failure_category='feature_engineering_strategy',
    difficulty='medium'
)

# ── HT-003: Feature importance interpretation ─────────────────────────────────
top_feats_str = '\n'.join([
    f'  {i+1:2d}. {f["feature"]}: {f["importance"]}'
    for i, f in enumerate(ground_truth['top_20_features'][:10])
])
logger.log(
    stage='Feature Selection',
    prompt='Based on the LightGBM feature importances below, what observations do you make and what actions would you take for a production credit risk model at a US bank?',
    data_context=f'Top 10 features by LightGBM importance:\n{top_feats_str}',
    ground_truth=(
        f'Key observations: '
        f'EXT_SOURCE_1/2/3 are external credit bureau scores — expected top predictors. '
        f'DAYS_BIRTH encodes applicant age — strong predictor BUT regulatory risk: '
        f'the Equal Credit Opportunity Act (ECOA) restricts using age in US credit decisions. '
        f'Must run disparate impact analysis before deploying features like DAYS_BIRTH. '
        f'CREDIT_INCOME_RATIO and domain ratio features confirm feature engineering was effective. '
        f'LightGBM importance score ≠ safe to deploy — compliance review required. '
        f'Gemini failure: says "keep all high-importance features" without mentioning ECOA or fair lending.'
    ),
    failure_category='domain_knowledge',
    difficulty='hard'
)

# ── HT-004: Bureau aggregation strategy ───────────────────────────────────────
logger.log(
    stage='Feature Engineering',
    prompt='Given the schema of bureau.csv, describe your complete aggregation strategy to create customer-level features for a credit default prediction model.',
    data_context=(
        'bureau.csv schema (one row per external loan, multiple loans per customer):\n'
        '  SK_ID_CURR: customer ID\n'
        '  CREDIT_ACTIVE: Active / Closed / Sold / Bad debt\n'
        '  AMT_CREDIT_SUM: total credit amount\n'
        '  AMT_CREDIT_SUM_DEBT: current debt\n'
        '  AMT_CREDIT_SUM_OVERDUE: overdue amount\n'
        '  CREDIT_DAY_OVERDUE: days past due\n'
        '  DAYS_CREDIT: days before application bureau record was created\n'
        '  bureau_balance.csv: monthly snapshots (STATUS: 0=current, 1-5=DPD, C=closed, X=unknown)\n\n'
        'Customers have between 1 and 40+ bureau entries.'
    ),
    ground_truth=(
        'Industry-level strategy: '
        '(1) Separate Active vs Closed loans — different default signals. '
        '(2) Overdue features: sum/max of AMT_CREDIT_SUM_OVERDUE and CREDIT_DAY_OVERDUE — top predictors. '
        '(3) Credit history length: min(DAYS_CREDIT) = oldest entry (furthest in the past). '
        '(4) Compute loan-level utilization (AMT_CREDIT_SUM_DEBT/AMT_CREDIT_SUM) BEFORE aggregating. '
        '(5) From bureau_balance: DPD ratio = months with status 1-5 / total months. '
        'Gemini failure: simple mean/sum across all loans without Active/Closed separation or overdue focus.'
    ),
    failure_category='feature_engineering_strategy',
    difficulty='hard'
)

# ── HT-005: Class imbalance handling ──────────────────────────────────────────
logger.log(
    stage='Modeling',
    prompt='The target variable has an 8% positive rate (highly imbalanced). What is the correct strategy for handling this imbalance in a production credit risk model at a bank?',
    data_context=(
        f'Training set:\n'
        f'  Total samples  : {len(y_train):,}\n'
        f'  Default (1)    : {int(y_train.sum()):,} ({pos_rate:.2%})\n'
        f'  No default (0) : {int((y_train==0).sum()):,} ({1-pos_rate:.2%})\n'
        f'  Imbalance ratio: {neg_pos_ratio}:1\n\n'
        f'Model: LightGBM gradient boosting\n'
        f'Deployment: real-time loan approval scoring, regulatory environment: Basel III, IFRS 9'
    ),
    ground_truth=(
        f'Correct approach: scale_pos_weight={neg_pos_ratio} in LightGBM (used in this pipeline). '
        f'Do NOT use SMOTE — it destroys probability calibration required by IFRS 9/Basel III for PD estimates. '
        f'Do NOT use class_weight="balanced" with tree models — scale_pos_weight is the correct parameter. '
        f'Evaluate with AUC-ROC={ground_truth["oof_auc"]:.4f}, KS={ground_truth["ks_stat"]:.4f}, Gini={ground_truth["gini"]:.4f}. '
        f'NOT accuracy (predicting all negatives = {1-pos_rate:.2%} accuracy — misleading). '
        f'NOT F1 (requires threshold choice, ignores cost asymmetry). '
        f'Gemini failure: suggests SMOTE and optimizing F1.'
    ),
    failure_category='model_selection',
    difficulty='hard'
)

# ── HT-006: Credit risk metrics ───────────────────────────────────────────────
logger.log(
    stage='Evaluation',
    prompt='What metrics should be reported for this credit default model for the bank\'s risk management team? List all required metrics with their values.',
    data_context=(
        f'Model: LightGBM binary classifier\n'
        f'OOF AUC-ROC: {ground_truth["oof_auc"]:.5f}\n'
        f'Regulatory environment: Basel III, IFRS 9\n'
        f'Use case: Probability of Default (PD) model for retail lending'
    ),
    ground_truth=(
        f'Required credit risk industry metrics:\n'
        f'1. KS Statistic = {ground_truth["ks_stat"]:.4f} ({ground_truth["ks_stat"]*100:.1f}%) — PRIMARY metric\n'
        f'2. Gini = {ground_truth["gini"]:.4f} = 2×AUC−1\n'
        f'3. AUC-ROC = {ground_truth["oof_auc"]:.5f}\n'
        f'4. Default rate by score decile (monotonicity check)\n'
        f'5. PSI (Population Stability Index) for production monitoring\n'
        f'6. Lift chart\n'
        f'NOT reported: accuracy, F1, precision, recall — these are not used in credit risk validation. '
        f'Gemini failure: reports standard ML metrics (accuracy, F1) and omits KS, Gini, PSI.'
    ),
    failure_category='domain_knowledge',
    difficulty='hard'
)

# ── HT-007: Threshold selection ───────────────────────────────────────────────
cm_05 = ground_truth.get('cm_threshold_05', [[0,0],[0,0]])
logger.log(
    stage='Evaluation',
    prompt='At what probability threshold should you classify a loan applicant as high risk (likely to default)? Justify your answer with the data below.',
    data_context=(
        f'Model predicted probability distribution (LightGBM OOF):\n'
        f'  Mean score : {float(lgb_oof.mean()):.4f}\n'
        f'  Scores > 0.5: {int((lgb_oof > 0.5).sum()):,} applicants ({float((lgb_oof > 0.5).mean()):.2%})\n'
        f'  Scores > 0.1: {int((lgb_oof > 0.1).sum()):,} applicants ({float((lgb_oof > 0.1).mean()):.2%})\n\n'
        f'Confusion matrix at threshold = 0.5:\n'
        f'  TN={cm_05[0][0]:,}  FP={cm_05[0][1]:,}\n'
        f'  FN={cm_05[1][0]:,}  TP={cm_05[1][1]:,}\n'
        f'  Approval rate at 0.5: {ground_truth["approval_rate_05"]:.1%}\n\n'
        f'Business costs: approving a defaulter = $5,000 loss | rejecting good customer = $500 lost revenue'
    ),
    ground_truth=(
        f'Correct threshold: {ground_truth["optimal_threshold"]:.3f} (NOT 0.5). '
        f'At threshold=0.5: approval rate={ground_truth["approval_rate_05"]:.1%} — '
        f'nearly everyone approved because base rate is only {pos_rate:.1%} and model rarely predicts >50%. '
        f'At optimal threshold={ground_truth["optimal_threshold"]:.3f}: '
        f'approval rate={ground_truth["approval_rate_optimal"]:.1%} with minimized $5,000/$500 cost. '
        f'Threshold is set by business cost matrix, not by model statistics. '
        f'Gemini failure: answers "use 0.5" without recognizing it is inappropriate for low base rate.'
    ),
    failure_category='metric_selection',
    difficulty='hard'
)

# ── HT-008: EXT_SOURCE features ───────────────────────────────────────────────
ext_miss = {
    col: float(app_train[col].isnull().mean()) if col in app_train.columns else 0.0
    for col in ['EXT_SOURCE_1', 'EXT_SOURCE_2', 'EXT_SOURCE_3']
}
ext_imp_str = '\n'.join([
    f'  {f["feature"]}: importance={f["importance"]}'
    for f in ground_truth['top_20_features']
    if 'EXT_SOURCE' in f['feature']
])
logger.log(
    stage='Feature Engineering',
    prompt='EXT_SOURCE_1, EXT_SOURCE_2, EXT_SOURCE_3 are among the top predictors. What are these features and what is the correct strategy for using them?',
    data_context=(
        f'EXT_SOURCE features in application_train.csv:\n'
        f'  EXT_SOURCE_1: missing {ext_miss["EXT_SOURCE_1"]:.1%} of rows\n'
        f'  EXT_SOURCE_2: missing {ext_miss["EXT_SOURCE_2"]:.1%} of rows\n'
        f'  EXT_SOURCE_3: missing {ext_miss["EXT_SOURCE_3"]:.1%} of rows\n\n'
        f'LightGBM importance (from pipeline):\n{ext_imp_str}\n\n'
        f'Data dictionary description: "Normalized score from external data source"'
    ),
    ground_truth=(
        'EXT_SOURCE_1/2/3 are EXTERNAL CREDIT BUREAU SCORES from third-party agencies. '
        'Correct strategy: (1) EXT_SOURCE_1 has high missingness — create EXT_SOURCE_1_MISSING flag. '
        '(2) Create combined features: EXT_SCORE_MEAN, EXT_SCORE_MIN, EXT_SCORE_STD across all three. '
        '(3) Interaction: EXT_SCORE_MEAN × CREDIT_INCOME_RATIO captures affordability vs creditworthiness. '
        '(4) LightGBM handles NaN natively — no imputation needed for tree models. '
        'Gemini failure: treats as anonymous numeric features without identifying them as bureau scores; '
        'misses the combined score and interaction features.'
    ),
    failure_category='domain_knowledge',
    difficulty='medium'
)

# ── HT-009: Cross-validation strategy ─────────────────────────────────────────
logger.log(
    stage='Modeling',
    prompt='A colleague suggests KFold(n_splits=5, shuffle=True) for cross-validation. What is your recommendation?',
    data_context=(
        f'Dataset: {len(y_train):,} samples, {pos_rate:.2%} positive rate\n'
        f'Expected samples per fold: ~{len(y_train)//5:,}\n'
        f'Expected positives per fold with random KFold: ~{int(len(y_train)*pos_rate/5):,} '
        f'(but can vary ±{int(len(y_train)*pos_rate/5 * 0.3):.0f} due to randomness)\n'
        f'Temporal note: applications were collected over multiple years (DAYS_DECISION range spans years)'
    ),
    ground_truth=(
        'Use StratifiedKFold — guarantees each fold has the same 8.07% positive rate. '
        'With random KFold and 8% base rate, fold positive rates can vary from 6-10% by chance, '
        'creating CV score variance unrelated to model quality. '
        'Deeper consideration Gemini misses: temporal ordering matters for loan models. '
        'Walk-forward validation (train on earlier loans, validate on later) better simulates deployment. '
        'StratifiedKFold ignores application date ordering — can introduce temporal leakage. '
        'Gemini usually gets the basic StratifiedKFold answer right but misses the temporal argument.'
    ),
    failure_category='model_selection',
    difficulty='medium'
)

# ── HT-010: Structural missingness after join ─────────────────────────────────
bur_cols = [c for c in train_master.columns if c.startswith('BUR_')]
no_bur_pct = float(train_master[bur_cols].isnull().any(axis=1).mean()) if bur_cols else 0.15
logger.log(
    stage='Preprocessing',
    prompt='After left-joining bureau features onto the application table, a portion of customers have NaN in all bureau features. How should these NaN values be handled?',
    data_context=(
        f'After left join of bureau aggregations:\n'
        f'  Customers with no bureau history (all BUR_* = NaN): ~{no_bur_pct:.1%} of training set\n'
        f'  (~{int(no_bur_pct * len(train_master)):,} customers have never had an external loan)\n\n'
        f'Available approaches: impute with 0, -1, column median, column mean, or separate indicator'
    ),
    ground_truth=(
        'This is STRUCTURAL missingness, not random missing data. '
        'NaN means the customer has ZERO external loans — not that data was lost. '
        'Correct strategy: (1) Impute count/sum features with 0 (zero history = zero count). '
        '(2) Create NO_BUREAU_HISTORY binary flag — thin-file customers have distinct default risk. '
        '(3) Do NOT impute ratios with median — treats no-history same as median-history customers. '
        'The no-history flag is predictive: first-time borrowers default differently than experienced ones. '
        'Gemini failure: suggests median imputation without recognizing structural nature of missingness '
        'and without creating the no-history indicator.'
    ),
    failure_category='data_leakage',
    difficulty='medium'
)

print(f'\nPhase 2: {len(logger.tasks)} headroom tasks created from real pipeline outputs.')

In [ ]:

# ── query_gemini: text-only helper for HT headroom tasks ─────────────────────
# (client, GEMINI_MODEL, CLAUDE_MODEL already defined in 1kp708fammd)

def query_gemini(prompt: str, data_context: str, retries: int = 3) -> str:
    '''
    Simple text completion for Gemini via OpenRouter.
    Used for the targeted headroom tasks (HT-001..010) — no code execution needed.
    '''
    full_prompt = (
        'You are a senior data scientist working on an end-to-end ML project '
        'using the Home Credit Default Risk dataset.\n\n'
        f'DATA CONTEXT (real pipeline output):\n{data_context}\n\n'
        f'QUESTION:\n{prompt}'
    )
    for attempt in range(retries):
        try:
            response = client.chat.completions.create(
                model    = GEMINI_MODEL,
                messages = [{'role': 'user', 'content': full_prompt}],
                max_tokens  = 1024,
                temperature = 0.0
            )
            return response.choices[0].message.content.strip()
        except Exception as e:
            print(f'  Attempt {attempt+1} failed: {e}')
            time.sleep(3)
    return 'ERROR: Failed after retries'


def query_claude(prompt: str, data_context: str, retries: int = 3) -> str:
    '''
    Simple text completion for Claude via OpenRouter.
    Used for the targeted headroom tasks (HT-001..010) — no code execution needed.
    '''
    full_prompt = (
        'You are a senior data scientist working on an end-to-end ML project '
        'using the Home Credit Default Risk dataset.\n\n'
        f'DATA CONTEXT (real pipeline output):\n{data_context}\n\n'
        f'QUESTION:\n{prompt}'
    )
    for attempt in range(retries):
        try:
            response = client.chat.completions.create(
                model    = CLAUDE_MODEL,
                messages = [{'role': 'user', 'content': full_prompt}],
                max_tokens  = 1024,
                temperature = 0.0
            )
            return response.choices[0].message.content.strip()
        except Exception as e:
            print(f'  Attempt {attempt+1} failed: {e}')
            time.sleep(3)
    return 'ERROR: Failed after retries'


print(f'query_gemini() and query_claude() ready (text-only, for HT tasks).')


In [ ]:

# ── Run HT headroom tasks on BOTH Gemini and Claude (text-only) ───────────────
print('Running HT headroom tasks (HT-001..010) on Gemini...\n')
for i, task in enumerate(logger.tasks):
    print(f'  {task["task_id"]} ({i+1}/{len(logger.tasks)}) — {task["stage"]}...')
    response = query_gemini(task['prompt'], task['data_context'])
    logger.tasks[i]['gemini_response'] = response
    time.sleep(1)
print(f'Gemini: {len(logger.tasks)} tasks done.\n')

print('Running HT headroom tasks (HT-001..010) on Claude...\n')
for i, task in enumerate(logger.tasks):
    print(f'  {task["task_id"]} ({i+1}/{len(logger.tasks)}) — {task["stage"]}...')
    response = query_claude(task['prompt'], task['data_context'])
    logger.tasks[i]['claude_response'] = response
    time.sleep(1)
print(f'Claude: {len(logger.tasks)} tasks done.')


In [ ]:

# ── Review: Gemini vs Claude on HT headroom tasks ─────────────────────────────
headroom_df = logger.to_dataframe()

for _, row in headroom_df.iterrows():
    print(f'\n{"="*70}')
    print(f'Task     : {row["task_id"]}  |  Stage: {row["stage"]}')
    print(f'Category : {row["failure_category"]}  |  Difficulty: {row["difficulty"]}')
    print(f'\nDATA CONTEXT:\n{str(row["data_context"])[:200]}...')
    print(f'\nGROUND TRUTH:\n{str(row["ground_truth"])[:200]}')
    print(f'\nGEMINI RESPONSE:\n{str(row.get("gemini_response",""))[:200]}')
    print(f'\nCLAUDE RESPONSE:\n{str(row.get("claude_response",""))[:200]}')


---
## Phase 2c: Hard Analytical Questions — Both Models

10 questions that require **multi-table joins, statistical computation, and code execution**
to answer correctly. These are deliberately designed to be hard enough to fail Gemini.

| Task | Topic | What makes it hard |
|---|---|---|
| HQ-001 | Consecutive overdue detection | Requires time-series pattern detection in bureau_balance |
| HQ-002 | Laborer payment behavior | 3-table join + conditional aggregation |
| HQ-003 | PSI computation | Statistical formula + 10-bin distribution comparison |
| HQ-004 | Partial correlation | Controlling for confounders — advanced stats |
| HQ-005 | Recency-weighted bureau features | Custom aggregation + AUC comparison |
| HQ-006 | Default rate by prev app count | Non-obvious non-monotonic relationship |
| HQ-007 | CREDIT_TYPE predictive power | Per-segment AUC computation |
| HQ-008 | DAYS_EMPLOYED anomaly investigation | Two-table join + t-test |
| HQ-009 | Gini vs KS derivation | Mathematical relationship between credit metrics |
| HQ-010 | Highest-risk income segment | Multi-criteria segment ranking |

Ground truths are pre-computed from the LightGBM pipeline before running either model.


In [ ]:

# ── Pre-compute ground truth answers for all 10 hard questions ─────────────────
# These are the correct answers — computed from the actual data.
# Models are scored against these.

from scipy import stats
import numpy as np

ground_truth_hq = {}

# HQ-001: Consecutive overdue months in bureau_balance
try:
    bb = bureau_bal.copy()
    bb['overdue'] = bb['STATUS'].isin(['1','2','3','4','5']).astype(int)
    bb = bb.sort_values(['SK_ID_BUREAU','MONTHS_BALANCE'])
    # consecutive overdue: cumsum reset trick
    def max_consec(s):
        cs = s.cumsum()
        reset = cs - cs.where(s==0).ffill().fillna(0)
        return reset.max()
    bb_consec = bb.groupby('SK_ID_BUREAU')['overdue'].apply(max_consec).reset_index()
    bb_consec.columns = ['SK_ID_BUREAU', 'max_consec_overdue']
    bb_app = bureau[['SK_ID_CURR','SK_ID_BUREAU']].merge(bb_consec, on='SK_ID_BUREAU')
    bb_app_max = bb_app.groupby('SK_ID_CURR')['max_consec_overdue'].max()
    pct_3consec = (bb_app_max >= 3).mean()
    ground_truth_hq['hq001_consecutive_overdue_pct'] = f'{pct_3consec:.1%} of applicants had ≥3 consecutive overdue months'
    print(f'HQ-001: {pct_3consec:.1%} of applicants had ≥3 consecutive overdue months')
except Exception as e:
    ground_truth_hq['hq001_consecutive_overdue_pct'] = f'Error: {e}'
    print(f'HQ-001 error: {e}')

# HQ-002: Laborers late payment rate vs default rate
try:
    inst_late = installments.copy()
    inst_late['days_late'] = inst_late['DAYS_ENTRY_PAYMENT'] - inst_late['DAYS_INSTALMENT']
    avg_late = inst_late.groupby('SK_ID_CURR')['days_late'].mean().reset_index()
    avg_late.columns = ['SK_ID_CURR', 'avg_days_late']
    laborers = app_train[app_train['OCCUPATION_TYPE']=='Laborers'][['SK_ID_CURR','TARGET']]
    lab_late = laborers.merge(avg_late, on='SK_ID_CURR', how='left')
    pct_late30 = (lab_late['avg_days_late'] > 30).mean()
    lab_default = laborers['TARGET'].mean()
    nonlab_default = app_train[app_train['OCCUPATION_TYPE']!='Laborers']['TARGET'].mean()
    ground_truth_hq['hq002_laborer_late_pct'] = (
        f'{pct_late30:.1%} of Laborers paid >30 days late on average. '
        f'Laborer default rate: {lab_default:.2%} vs non-Laborer: {nonlab_default:.2%}'
    )
    print(f'HQ-002: {pct_late30:.1%} Laborers >30d late; default {lab_default:.2%} vs {nonlab_default:.2%}')
except Exception as e:
    ground_truth_hq['hq002_laborer_late_pct'] = f'Error: {e}'

# HQ-003: PSI for top 5 features
try:
    def compute_psi(train_vals, test_vals, bins=10):
        train_vals = train_vals.dropna()
        test_vals  = test_vals.dropna()
        breakpoints = np.percentile(train_vals, np.linspace(0, 100, bins+1))
        breakpoints = np.unique(breakpoints)
        tr_cnt = np.histogram(train_vals, bins=breakpoints)[0]
        te_cnt = np.histogram(test_vals,  bins=breakpoints)[0]
        tr_pct = (tr_cnt / tr_cnt.sum()).clip(1e-6)
        te_pct = (te_cnt / te_cnt.sum()).clip(1e-6)
        return float(np.sum((te_pct - tr_pct) * np.log(te_pct / tr_pct)))

    top5 = [f['feature'] for f in ground_truth.get('top_20_features', [])[:5]]
    psi_results = {}
    for feat in top5:
        if feat in app_train.columns and feat in app_test.columns:
            psi_val = compute_psi(app_train[feat], app_test[feat])
            psi_results[feat] = round(psi_val, 4)
    ground_truth_hq['hq003_psi_values'] = str(psi_results)
    print(f'HQ-003 PSI: {psi_results}')
except Exception as e:
    ground_truth_hq['hq003_psi_values'] = f'Error: {e}'

# HQ-004: Partial correlations EXT_SOURCE vs TARGET controlling AMT_CREDIT
try:
    from scipy.stats import pearsonr
    df_pc = app_train[['EXT_SOURCE_1','EXT_SOURCE_2','EXT_SOURCE_3','TARGET','AMT_CREDIT']].dropna()
    def partial_corr(x, y, z):
        rx_z = pearsonr(x, z)[0]; ry_z = pearsonr(y, z)[0]; rxy = pearsonr(x, y)[0]
        return (rxy - rx_z*ry_z) / (np.sqrt(1-rx_z**2) * np.sqrt(1-ry_z**2) + 1e-9)
    pc12 = partial_corr(df_pc['EXT_SOURCE_1'], df_pc['TARGET'], df_pc['AMT_CREDIT'])
    pc23 = partial_corr(df_pc['EXT_SOURCE_2'], df_pc['TARGET'], df_pc['AMT_CREDIT'])
    pc13 = partial_corr(df_pc['EXT_SOURCE_3'], df_pc['TARGET'], df_pc['AMT_CREDIT'])
    ground_truth_hq['hq004_partial_corr'] = (
        f'Partial correlations with TARGET (controlling AMT_CREDIT): '
        f'EXT_SOURCE_1={pc12:.3f}, EXT_SOURCE_2={pc23:.3f}, EXT_SOURCE_3={pc13:.3f}. '
        f'Best interaction: EXT_SOURCE_2 × EXT_SOURCE_3'
    )
    print(f'HQ-004: pc1={pc12:.3f}, pc2={pc23:.3f}, pc3={pc13:.3f}')
except Exception as e:
    ground_truth_hq['hq004_partial_corr'] = f'Error: {e}'

# HQ-005: Recency-weighted bureau AUC (qualitative answer)
ground_truth_hq['hq005_recency_auc'] = (
    'Recency-weighted mean AMT_CREDIT_SUM (weight=exp(-0.1*abs(DAYS_CREDIT))) '
    'typically improves AUC by 0.001-0.003 over simple mean due to '
    'more recent loan behavior being more predictive of current default risk.'
)

# HQ-006: Default rate by number of previous applications
try:
    prev_cnt = prev_app.groupby('SK_ID_CURR').size().reset_index(name='n_prev')
    prev_cnt['n_prev_cat'] = prev_cnt['n_prev'].clip(upper=4).map(
        {0:'0',1:'1',2:'2',3:'3',4:'4+'}).fillna('4+')
    app_prev = app_train[['SK_ID_CURR','TARGET']].merge(prev_cnt, on='SK_ID_CURR', how='left')
    app_prev['n_prev_cat'] = app_prev['n_prev_cat'].fillna('0')
    dr_by_prev = app_prev.groupby('n_prev_cat')['TARGET'].mean().round(4).to_dict()
    ground_truth_hq['hq006_default_by_prev_count'] = (
        f'Default rate by prev app count: {dr_by_prev}. '
        f'Relationship is non-monotonic — applicants with 1-2 prev apps have lower default than 0 or 4+.'
    )
    print(f'HQ-006: {dr_by_prev}')
except Exception as e:
    ground_truth_hq['hq006_default_by_prev_count'] = f'Error: {e}'

# HQ-007: Top 3 CREDIT_TYPE by AUC (simplified: by default rate difference)
try:
    from sklearn.metrics import roc_auc_score
    bur_app = bureau[['SK_ID_CURR','CREDIT_TYPE']].merge(
        app_train[['SK_ID_CURR','TARGET']], on='SK_ID_CURR')
    ct_aucs = {}
    for ct in bur_app['CREDIT_TYPE'].value_counts().head(6).index:
        flag = bur_app.groupby('SK_ID_CURR').apply(
            lambda x: int((x['CREDIT_TYPE']==ct).any())).reset_index(name='flag')
        merged = app_train[['SK_ID_CURR','TARGET']].merge(flag, on='SK_ID_CURR', how='left').fillna(0)
        try:
            auc = roc_auc_score(merged['TARGET'], merged['flag'])
            ct_aucs[ct] = round(auc, 4)
        except:
            pass
    top3_ct = sorted(ct_aucs.items(), key=lambda x: abs(x[1]-0.5), reverse=True)[:3]
    ground_truth_hq['hq007_credit_type_auc'] = f'Top 3 CREDIT_TYPE by AUC: {top3_ct}'
    print(f'HQ-007: {top3_ct}')
except Exception as e:
    ground_truth_hq['hq007_credit_type_auc'] = f'Error: {e}'

# HQ-008: Credit utilization t-test for DAYS_EMPLOYED anomaly
try:
    anom_ids = set(app_train[app_train['DAYS_EMPLOYED']==365243]['SK_ID_CURR'])
    bur_util = bureau.copy()
    bur_util['utilization'] = (bur_util['AMT_CREDIT_SUM_DEBT'] / bur_util['AMT_CREDIT_SUM']).clip(0,2)
    util_app = bur_util.groupby('SK_ID_CURR')['utilization'].mean().reset_index()
    util_app['is_anom'] = util_app['SK_ID_CURR'].isin(anom_ids)
    anom_util  = util_app[util_app['is_anom']]['utilization'].dropna()
    norm_util  = util_app[~util_app['is_anom']]['utilization'].dropna()
    t_stat, p_val = stats.ttest_ind(anom_util, norm_util)
    ground_truth_hq['hq008_utilization_ttest'] = (
        f'Anomaly applicants avg utilization={anom_util.mean():.3f}, '
        f'normal={norm_util.mean():.3f}, t={t_stat:.2f}, p={p_val:.4f} '
        f'({"significant" if p_val < 0.05 else "not significant"} at α=0.05)'
    )
    print(f'HQ-008: anom={anom_util.mean():.3f}, normal={norm_util.mean():.3f}, p={p_val:.4f}')
except Exception as e:
    ground_truth_hq['hq008_utilization_ttest'] = f'Error: {e}'

# HQ-009: Gini vs KS relationship
oof_auc = ground_truth.get('oof_auc', 0.795)
gini    = ground_truth.get('gini',    round(2*oof_auc-1, 4))
ks      = ground_truth.get('ks_stat', 'TBD')
ground_truth_hq['hq009_gini_ks_derivation'] = (
    f'Gini = 2*AUC - 1 = 2*{oof_auc:.4f}-1 = {gini:.4f}. '
    f'KS={ks} measures max separation between TPR and FPR curves. '
    f'KS ≠ Gini because KS uses the maximum point on the ROC curve '
    f'while Gini integrates the full area. KS is more sensitive to the '
    f'best single threshold; Gini measures overall discrimination.'
)

# HQ-010: NAME_INCOME_TYPE segment analysis
try:
    seg = app_train.groupby('NAME_INCOME_TYPE').agg(
        default_rate=('TARGET','mean'),
        avg_credit=('AMT_CREDIT','mean'),
        total_volume=('AMT_CREDIT','sum'),
        count=('TARGET','count')
    ).round(4)
    seg['risk_rank'] = seg['default_rate'].rank(ascending=False) + seg['avg_credit'].rank(ascending=False)
    top_seg = seg['risk_rank'].idxmin()
    ground_truth_hq['hq010_income_segment'] = (
        f'Highest risk segment: {top_seg} — '
        f'default_rate={seg.loc[top_seg,"default_rate"]:.2%}, '
        f'avg_credit={seg.loc[top_seg,"avg_credit"]:,.0f}'
    )
    print(f'HQ-010: {top_seg} — {seg.loc[top_seg,"default_rate"]:.2%} default, {seg.loc[top_seg,"avg_credit"]:,.0f} avg credit')
except Exception as e:
    ground_truth_hq['hq010_income_segment'] = f'Error: {e}'

print(f'\nGround truth computed for {len(ground_truth_hq)} HQ tasks.')


In [ ]:

# ── Define 10 hard analytical questions ───────────────────────────────────────
# These require multi-table joins, domain knowledge, and code execution to answer.
# Ground truth is pre-computed from the LightGBM pipeline.

hq_tasks = [
    {
        'task_id': 'HQ-001',
        'stage': 'EDA / Bureau',
        'difficulty': 'hard',
        'failure_category': 'temporal_pattern_detection',
        'prompt': (
            'Using the bureau_balance table, compute the proportion of loan applicants '
            '(SK_ID_CURR) who had at least 3 consecutive months with STATUS in ("1","2","3","4","5") '
            '(overdue) in their most recent bureau loan. '
            'Show your SQL-equivalent logic in pandas.'
        ),
        'data_context': (
            f'bureau_bal has {len(bureau_bal):,} rows. Columns: SK_ID_BUREAU, MONTHS_BALANCE, STATUS.\n'
            f'STATUS values: 0=no_loan, C=paid, X=unknown, 1-5=overdue_months.\n'
            f'bureau links SK_ID_CURR to SK_ID_BUREAU ({len(bureau):,} rows).\n'
            f'Total unique applicants: {bureau["SK_ID_CURR"].nunique():,}'
        ),
        'ground_truth': ground_truth_hq.get('hq001_consecutive_overdue_pct', 'TBD'),
        'scoring_keywords': ['consecutive', 'overdue', 'proportion', '%', 'status']
    },
    {
        'task_id': 'HQ-002',
        'stage': 'Feature Engineering / Installments',
        'difficulty': 'hard',
        'failure_category': 'multi_table_aggregation',
        'prompt': (
            'For applicants with OCCUPATION_TYPE="Laborers", compute the percentage who '
            'paid their installments more than 30 days late on average. '
            'Late days = DAYS_ENTRY_PAYMENT - DAYS_INSTALMENT (positive = late). '
            'Compare their default rate to non-Laborers.'
        ),
        'data_context': (
            f'installments has {len(installments):,} rows: SK_ID_CURR, SK_ID_PREV, '
            f'DAYS_INSTALMENT, DAYS_ENTRY_PAYMENT, AMT_INSTALMENT, AMT_PAYMENT.\n'
            f'app_train has OCCUPATION_TYPE with {app_train["OCCUPATION_TYPE"].nunique()} unique values.\n'
            f'Laborer count: {(app_train["OCCUPATION_TYPE"]=="Laborers").sum():,}'
        ),
        'ground_truth': ground_truth_hq.get('hq002_laborer_late_pct', 'TBD'),
        'scoring_keywords': ['laborer', 'late', 'days', 'default', 'percent']
    },
    {
        'task_id': 'HQ-003',
        'stage': 'Model Validation / Distribution Shift',
        'difficulty': 'very_hard',
        'failure_category': 'statistical_computation',
        'prompt': (
            'Compute the Population Stability Index (PSI) for the top 5 features by '
            'LightGBM importance between the train and test distributions. '
            'PSI = sum((actual% - expected%) * ln(actual%/expected%)) using 10 bins. '
            'Flag any feature with PSI > 0.2 as unstable.'
        ),
        'data_context': (
            f'Top 5 features by LightGBM importance: '
            f'{", ".join([f["feature"] for f in ground_truth.get("top_20_features", [])[:5]])}.\n'
            f'Train: {len(app_train):,} rows | Test: {len(app_test):,} rows.\n'
            f'PSI thresholds: <0.1 stable, 0.1-0.2 minor shift, >0.2 major shift.'
        ),
        'ground_truth': ground_truth_hq.get('hq003_psi_values', 'TBD'),
        'scoring_keywords': ['psi', 'stable', 'shift', '0.1', '0.2']
    },
    {
        'task_id': 'HQ-004',
        'stage': 'EDA / Feature Interaction',
        'difficulty': 'very_hard',
        'failure_category': 'statistical_reasoning',
        'prompt': (
            'Compute the partial correlation between each pair of (EXT_SOURCE_1, EXT_SOURCE_2, '
            'EXT_SOURCE_3) and TARGET, controlling for AMT_CREDIT. '
            'Which pair shows the strongest combined predictive power? '
            'Suggest a feature interaction term.'
        ),
        'data_context': (
            f'EXT_SOURCE missingness: '
            f'EXT_SOURCE_1={app_train["EXT_SOURCE_1"].isnull().mean():.1%}, '
            f'EXT_SOURCE_2={app_train["EXT_SOURCE_2"].isnull().mean():.1%}, '
            f'EXT_SOURCE_3={app_train["EXT_SOURCE_3"].isnull().mean():.1%}.\n'
            f'Correlations with TARGET: '
            f'EXT_SOURCE_1={app_train["EXT_SOURCE_1"].corr(app_train["TARGET"]):.3f}, '
            f'EXT_SOURCE_2={app_train["EXT_SOURCE_2"].corr(app_train["TARGET"]):.3f}, '
            f'EXT_SOURCE_3={app_train["EXT_SOURCE_3"].corr(app_train["TARGET"]):.3f}'
        ),
        'ground_truth': ground_truth_hq.get('hq004_partial_corr', 'TBD'),
        'scoring_keywords': ['partial', 'correlation', 'interaction', 'ext_source', 'combined']
    },
    {
        'task_id': 'HQ-005',
        'stage': 'Feature Engineering / Bureau Recency',
        'difficulty': 'very_hard',
        'failure_category': 'advanced_aggregation',
        'prompt': (
            'Build a recency-weighted aggregation of AMT_CREDIT_SUM from the bureau table. '
            'Weight = exp(-0.1 * abs(DAYS_CREDIT)) so more recent loans get higher weight. '
            'Compare the AUC of a simple LightGBM with: (a) raw mean AMT_CREDIT_SUM, '
            '(b) recency-weighted mean. Does recency weighting improve AUC?'
        ),
        'data_context': (
            f'bureau: {len(bureau):,} rows. DAYS_CREDIT range: '
            f'[{bureau["DAYS_CREDIT"].min()}, {bureau["DAYS_CREDIT"].max()}].\n'
            f'AMT_CREDIT_SUM: mean={bureau["AMT_CREDIT_SUM"].mean():.0f}, '
            f'null%={bureau["AMT_CREDIT_SUM"].isnull().mean():.1%}.\n'
            f'Baseline LightGBM OOF AUC (all features): {ground_truth.get("oof_auc", "TBD")}'
        ),
        'ground_truth': ground_truth_hq.get('hq005_recency_auc', 'TBD'),
        'scoring_keywords': ['recency', 'weight', 'auc', 'exp', 'decay']
    },
    {
        'task_id': 'HQ-006',
        'stage': 'EDA / Previous Applications',
        'difficulty': 'hard',
        'failure_category': 'multi_table_join_analysis',
        'prompt': (
            'What is the default rate (TARGET=1 rate) for applicants grouped by the NUMBER '
            'of previous loan applications they have (0, 1, 2, 3, 4+)? '
            'Is there a monotonic relationship? What does this imply about feature engineering?'
        ),
        'data_context': (
            f'prev_app: {len(prev_app):,} rows, {prev_app["SK_ID_CURR"].nunique():,} unique applicants.\n'
            f'app_train: {len(app_train):,} rows, default rate={app_train["TARGET"].mean():.2%}.\n'
            f'Applicants with 0 previous apps: '
            f'{app_train["SK_ID_CURR"].nunique() - prev_app["SK_ID_CURR"].nunique():,}'
        ),
        'ground_truth': ground_truth_hq.get('hq006_default_by_prev_count', 'TBD'),
        'scoring_keywords': ['previous', 'count', 'default', 'rate', 'monotonic']
    },
    {
        'task_id': 'HQ-007',
        'stage': 'Feature Engineering / Credit Type',
        'difficulty': 'hard',
        'failure_category': 'per_segment_auc',
        'prompt': (
            'From the bureau table, identify the top 3 CREDIT_TYPE values that best '
            'predict default (TARGET). For each type, compute: '
            '(1) count of applicants who have that credit type, '
            '(2) default rate among those applicants, '
            '(3) AUC if you used a binary flag for that credit type as sole feature.'
        ),
        'data_context': (
            f'bureau CREDIT_TYPE unique values: {bureau["CREDIT_TYPE"].nunique()}.\n'
            f'Top 3 by frequency: {bureau["CREDIT_TYPE"].value_counts().head(3).to_dict()}.\n'
            f'bureau links to app_train via SK_ID_CURR.'
        ),
        'ground_truth': ground_truth_hq.get('hq007_credit_type_auc', 'TBD'),
        'scoring_keywords': ['credit_type', 'auc', 'default', 'flag', 'predict']
    },
    {
        'task_id': 'HQ-008',
        'stage': 'EDA / DAYS_EMPLOYED Anomaly',
        'difficulty': 'hard',
        'failure_category': 'anomaly_investigation',
        'prompt': (
            'For applicants flagged with DAYS_EMPLOYED=365243 (the anomaly), compute their '
            'average bureau credit utilization (AMT_CREDIT_SUM_DEBT / AMT_CREDIT_SUM) '
            'vs. non-flagged applicants. Run a t-test. Is the difference statistically significant?'
        ),
        'data_context': (
            f'DAYS_EMPLOYED=365243 count: {ground_truth.get("days_employed_anom_count", "TBD")} '
            f'({ground_truth.get("days_employed_anom_pct", "TBD"):.1%} of train).\n'
            f'bureau: AMT_CREDIT_SUM_DEBT null%={bureau["AMT_CREDIT_SUM_DEBT"].isnull().mean():.1%}, '
            f'AMT_CREDIT_SUM null%={bureau["AMT_CREDIT_SUM"].isnull().mean():.1%}.'
        ),
        'ground_truth': ground_truth_hq.get('hq008_utilization_ttest', 'TBD'),
        'scoring_keywords': ['utilization', 'ttest', 'significant', 'anomaly', 'p-value']
    },
    {
        'task_id': 'HQ-009',
        'stage': 'Model Evaluation / Metrics',
        'difficulty': 'very_hard',
        'failure_category': 'metric_derivation',
        'prompt': (
            'Using the LightGBM OOF predictions, mathematically derive the relationship '
            'between Gini coefficient and KS statistic. '
            'Show: (1) Gini = 2*AUC - 1, (2) compute KS from the OOF predictions, '
            '(3) explain why KS ≠ Gini despite both measuring discrimination power.'
        ),
        'data_context': (
            f'OOF AUC = {ground_truth.get("oof_auc", "TBD")}, '
            f'Gini = {ground_truth.get("gini", "TBD")}, '
            f'KS = {ground_truth.get("ks_stat", "TBD")}.\n'
            f'OOF predictions are in `lgb_oof` (numpy array, {len(app_train):,} values).\n'
            f'True labels are in `y_train`.'
        ),
        'ground_truth': ground_truth_hq.get('hq009_gini_ks_derivation', 'TBD'),
        'scoring_keywords': ['gini', 'ks', 'auc', '2*auc-1', 'derivation']
    },
    {
        'task_id': 'HQ-010',
        'stage': 'EDA / Segment Analysis',
        'difficulty': 'hard',
        'failure_category': 'segment_prioritization',
        'prompt': (
            'Identify which NAME_INCOME_TYPE segment simultaneously has the HIGHEST default '
            'rate AND the HIGHEST average loan amount (AMT_CREDIT). '
            'This segment represents the highest business risk. '
            'Quantify: default rate, avg loan amount, and total loan volume for each segment.'
        ),
        'data_context': (
            f'NAME_INCOME_TYPE values: {app_train["NAME_INCOME_TYPE"].value_counts().to_dict()}.\n'
            f'AMT_CREDIT: mean={app_train["AMT_CREDIT"].mean():.0f}, '
            f'median={app_train["AMT_CREDIT"].median():.0f}.\n'
            f'Overall default rate: {app_train["TARGET"].mean():.2%}.'
        ),
        'ground_truth': ground_truth_hq.get('hq010_income_segment', 'TBD'),
        'scoring_keywords': ['income_type', 'default', 'highest', 'segment', 'risk']
    },
]

print(f'Defined {len(hq_tasks)} hard analytical questions.')
for hq in hq_tasks:
    print(f'  {hq["task_id"]} | {hq["stage"]} | {hq["difficulty"]}')


In [ ]:

# ── Run Gemini on all 10 hard questions (agentic, with code execution) ────────
print('Running 10 hard questions on Gemini (agentic)...\n')
gemini_hq_responses = {}

for hq in hq_tasks:
    qid = hq['task_id']
    print(f'Running {qid} — {hq["stage"]}...')
    resp, code_blocks = run_agent(
        model         = GEMINI_MODEL,
        system_prompt = HQ_SYSTEM,
        user_prompt   = (
            f'DATA CONTEXT:\n{hq["data_context"]}\n\n'
            f'QUESTION:\n{hq["prompt"]}\n\n'
            'Write and execute code to answer precisely. '
            'Print the exact numerical answer.'
        ),
        max_turns = 10,
        verbose   = False
    )
    text = resp.choices[0].message.content or ''
    gemini_hq_responses[qid] = text
    print(f'  Done. {len(code_blocks)} code blocks | {len(text)} chars response.')
    time.sleep(1)

print(f'\nAll {len(hq_tasks)} HQ tasks completed on Gemini.')


In [ ]:

# ── Run Claude Opus 4.6 on all 10 hard questions (agentic, with code execution) ─
HQ_SYSTEM = (
    'You are an expert data scientist. You have access to a Python environment '
    'with the Home Credit Default Risk dataset already loaded:\n'
    '  app_train, app_test, bureau, bureau_bal, prev_app, pos_cash, '
    'installments, credit_card\n'
    'Use execute_python to write and run code. Show exact numerical answers in your final response.'
)

print('Running 10 hard questions on Claude Opus 4.6 (agentic)...\n')
claude_hq_responses = {}

for hq in hq_tasks:
    qid = hq['task_id']
    print(f'Running {qid} — {hq["stage"]}...')
    resp, code_blocks = run_agent(
        model         = CLAUDE_MODEL,
        system_prompt = HQ_SYSTEM,
        user_prompt   = (
            f'DATA CONTEXT:\n{hq["data_context"]}\n\n'
            f'QUESTION:\n{hq["prompt"]}\n\n'
            'Write and execute code to answer precisely. '
            'Print the exact numerical answer.'
        ),
        max_turns = 10,
        verbose   = False
    )
    text = resp.choices[0].message.content or ''
    claude_hq_responses[qid] = text
    print(f'  Done. {len(code_blocks)} code blocks | {len(text)} chars response.')
    time.sleep(1)

print(f'\nAll {len(hq_tasks)} HQ tasks completed on Claude Opus 4.6.')


In [ ]:

# ── Score both models on HQ tasks ─────────────────────────────────────────────
hq_results = []
for hq in hq_tasks:
    qid      = hq['task_id']
    g_resp   = gemini_hq_responses.get(qid, '')
    c_resp   = claude_hq_responses.get(qid, '')
    gt       = hq['ground_truth']
    keywords = hq['scoring_keywords']

    g_correct = int(any(kw.lower() in g_resp.lower() for kw in keywords))
    c_correct = int(any(kw.lower() in c_resp.lower() for kw in keywords))

    hq_results.append({
        'task_id':          qid,
        'task_type':        'HQ',
        'stage':            hq['stage'],
        'difficulty':       hq['difficulty'],
        'failure_category': hq['failure_category'],
        'ground_truth':     gt[:200],
        'gemini_response':  g_resp[:200],
        'claude_response':  c_resp[:200],
        'gemini_correct':   g_correct,
        'claude_correct':   c_correct,
        'human_correct':    1,
        'scoring_keywords': ', '.join(keywords[:4]),
    })

hq_df = pd.DataFrame(hq_results)
print('Hard Question Results:')
print(hq_df[['task_id', 'stage', 'gemini_correct', 'claude_correct']].to_string(index=False))

gemini_hq_score = hq_df['gemini_correct'].mean()
claude_hq_score  = hq_df['claude_correct'].mean()
print(f'\nGemini HQ score: {gemini_hq_score:.0%}')
print(f'Claude  HQ score: {claude_hq_score:.0%}')
print(f'Headroom gap (HQ): Claude − Gemini = {(claude_hq_score - gemini_hq_score)*100:+.0f} pp')


In [ ]:
# ── Export predictions and model summary ──────────────────────────────────────
import json

submission = pd.DataFrame({
    'SK_ID_CURR': test_master['SK_ID_CURR'].astype(int),
    'TARGET':     lgb_test
})
submission.to_csv('/content/outputs/submission_lgbm.csv', index=False)

# Model performance summary
summary = {
    'dataset':        'Home Credit Default Risk',
    'model':          'LightGBM (5-fold StratifiedKFold CV)',
    'n_features':     len(selected_features),
    'oof_auc':        round(lgb_auc, 5),
    'gini':           round(gini, 5),
    'ks_statistic':   round(float(ks_lgbm), 5),
    'optimal_threshold': round(float(opt_threshold), 3),
    'n_headroom_tasks': len(headroom_df)
}

with open('/content/outputs/model_summary.json', 'w') as f:
    json.dump(summary, f, indent=2)

print('Model Summary:')
for k, v in summary.items():
    print(f'  {k:25s}: {v}')

---
## Phase 3: DSBench-Style Benchmark Results

Structured after **[DSBench (ICLR 2025)](https://github.com/LiqiangJing/DSBench)** — an evaluation framework for LLMs on Kaggle data science tasks.

| Concept | DSBench | This Project |
|---|---|---|
| Tasks | 540 (466 DA + 74 DM) | 10 data modeling tasks |
| Dataset source | Kaggle | Home Credit Default Risk (Kaggle) |
| Human baseline | Kaggle community best submissions | LightGBM 5-fold StratifiedKFold pipeline |
| Metric | Relative Performance Gap (RPG) | RPG = (human − model) / human × 100 |
| Models | GPT-4, Claude, Gemini, etc. | Gemini 2.0 Pro vs Claude Opus 4.6 |

**RPG interpretation**: 0% = matches human baseline. Higher = larger gap from human performance.
Tasks where **Gemini fails but Claude passes** are the core headroom dataset.


In [ ]:

# ── Combine HT + HQ tasks into unified DSBench benchmark_df ───────────────────
import pandas as pd, json

def score_task(ground_truth_str: str, response_str: str, keywords: list) -> int:
    resp_lower = (response_str or '').lower()
    return int(any(kw.lower() in resp_lower for kw in keywords if len(kw) > 3))

all_rows = []

# ── Headroom Tasks (HT-001..010) — text-only targeted DS decision questions ───
for task in logger.tasks:
    gemini_resp = task.get('gemini_response') or ''
    claude_resp = task.get('claude_response') or ''   # stored by moeprgcg02s
    gt_keywords = [w.strip('.,;:()').lower() for w in task['ground_truth'].split()
                   if len(w.strip('.,;:()')) > 4][:8]
    all_rows.append({
        'task_id':          task['task_id'],
        'task_type':        'HT',
        'dataset':          'Home Credit Default Risk',
        'source':           'Kaggle',
        'stage':            task['stage'],
        'difficulty':       task['difficulty'],
        'failure_category': task['failure_category'],
        'prompt':           task['prompt'][:250],
        'ground_truth':     task['ground_truth'][:300],
        'gemini_response':  gemini_resp[:300],
        'claude_response':  claude_resp[:300],
        'gemini_correct':   score_task(task['ground_truth'], gemini_resp, gt_keywords),
        'claude_correct':   score_task(task['ground_truth'], claude_resp, gt_keywords),
        'human_correct':    1,
    })

# ── Hard Questions (HQ-001..010) — agentic code execution required ─────────────
for row in hq_results:
    all_rows.append({
        'task_id':          row['task_id'],
        'task_type':        'HQ',
        'dataset':          'Home Credit Default Risk',
        'source':           'Kaggle',
        'stage':            row['stage'],
        'difficulty':       row['difficulty'],
        'failure_category': row['failure_category'],
        'prompt':           '',
        'ground_truth':     row['ground_truth'][:300],
        'gemini_response':  row['gemini_response'][:300],
        'claude_response':  row['claude_response'][:300],
        'gemini_correct':   row['gemini_correct'],
        'claude_correct':   row['claude_correct'],
        'human_correct':    1,
    })

benchmark_df = pd.DataFrame(all_rows)
ht_n = (benchmark_df['task_type'] == 'HT').sum()
hq_n = (benchmark_df['task_type'] == 'HQ').sum()
print(f'Benchmark: {len(benchmark_df)} tasks total ({ht_n} HT + {hq_n} HQ)')
print()
print(benchmark_df[['task_id','task_type','stage','gemini_correct','claude_correct']].to_string(index=False))


In [ ]:

# ── DSBench RPG metric + final comparison table (HT + HQ + Overall) ────────────
human_auc = ground_truth.get('oof_auc', 0.795)

def rpg(human, model):
    return (human - model) / human * 100 if human > 0 else 0.0

ht = benchmark_df[benchmark_df['task_type'] == 'HT']
hq = benchmark_df[benchmark_df['task_type'] == 'HQ']

gemini_ht = ht['gemini_correct'].mean(); claude_ht = ht['claude_correct'].mean()
gemini_hq = hq['gemini_correct'].mean(); claude_hq = hq['claude_correct'].mean()
gemini_all= benchmark_df['gemini_correct'].mean()
claude_all = benchmark_df['claude_correct'].mean()

sep = '=' * 66
print(sep)
print('  DSBENCH-STYLE BENCHMARK RESULTS — HOME CREDIT DEFAULT RISK')
print(sep)

comparison = pd.DataFrame({
    'Model': [
        'LightGBM (Human Baseline)',
        f'Gemini ({GEMINI_MODEL.split("/")[-1][:22]})',
        f'Claude ({CLAUDE_MODEL.split("/")[-1][:22]})',
    ],
    'HT Score': [
        '100%', f'{gemini_ht:.0%}', f'{claude_ht:.0%}'
    ],
    'HT RPG': [
        '0.0%', f'{rpg(1,gemini_ht):.1f}%', f'{rpg(1,claude_ht):.1f}%'
    ],
    'HQ Score': [
        '100%', f'{gemini_hq:.0%}', f'{claude_hq:.0%}'
    ],
    'HQ RPG': [
        '0.0%', f'{rpg(1,gemini_hq):.1f}%', f'{rpg(1,claude_hq):.1f}%'
    ],
    'Overall RPG': [
        '0.0%', f'{rpg(1,gemini_all):.1f}%', f'{rpg(1,claude_all):.1f}%'
    ],
    'OOF AUC': [
        f'{human_auc:.4f}',
        f'{claude_metrics.get("oof_auc","N/A")}',
        f'{claude_metrics.get("oof_auc","N/A")}',
    ]
})
print(comparison.to_string(index=False))
print(sep)

headroom = benchmark_df[
    (benchmark_df['gemini_correct'] == 0) & (benchmark_df['claude_correct'] == 1)
]
print(f'\nCore headroom tasks (Gemini FAIL, Claude PASS): {len(headroom)} / {len(benchmark_df)}')
for _, r in headroom.iterrows():
    print(f'  {r["task_id"]} [{r["task_type"]}] | {r["stage"]} | {r["failure_category"]}')


In [ ]:

# ── Export DSBench-compatible benchmark results ────────────────────────────────
benchmark_df.to_csv('/content/outputs/dsbench_benchmark.csv', index=False)
comparison.to_csv('/content/outputs/model_comparison.csv', index=False)

benchmark_meta = {
    'benchmark_name':    'Home Credit Headroom Dataset',
    'inspired_by':       'DSBench (ICLR 2025) — github.com/LiqiangJing/DSBench',
    'dataset':           'Home Credit Default Risk (Kaggle)',
    'n_tasks':           len(benchmark_df),
    'task_type':         'data_modeling',
    'models_evaluated':  [GEMINI_MODEL, CLAUDE_MODEL],
    'human_baseline':    'LightGBM 5-fold StratifiedKFold',
    'human_auc':         ground_truth.get('oof_auc', 'TBD'),
    'human_ks':          ground_truth.get('ks_stat', 'TBD'),
    'human_gini':        ground_truth.get('gini', 'TBD'),
    'gemini_task_score': float(benchmark_df['gemini_correct'].mean()),
    'claude_task_score': float(benchmark_df['claude_correct'].mean()),
    'gemini_rpg':        float(gemini_rpg),
    'claude_rpg':        float(claude_rpg),
    'gemini_analysis_pass_rate': float(gemini_analysis_score),
    'claude_analysis_pass_rate': float(claude_pass_rate),
    'rpg_formula':       '(human_score - model_score) / human_score * 100',
}

with open('/content/outputs/benchmark_meta.json', 'w') as f:
    json.dump(benchmark_meta, f, indent=2)

print('Exported benchmark results:')
print('  /content/outputs/dsbench_benchmark.csv  — task-level results (10 rows)')
print('  /content/outputs/model_comparison.csv   — model comparison table')
print('  /content/outputs/benchmark_meta.json    — benchmark metadata')
print(f'\nFinal: {len(benchmark_df)} tasks | Gemini RPG={gemini_rpg:.1f}% | Claude RPG={claude_rpg:.1f}%')


---
## 10. Project Report

### Overview
This project delivers two outputs from one end-to-end ML pipeline on the Home Credit Default Risk dataset:
1. **A production-grade credit default model** (LightGBM)
2. **A headroom benchmark dataset** — 10 tasks where Gemini Pro fails as a data scientist

---

### Pipeline Summary

| Stage | Actions | Key Decisions |
|---|---|---|
| **Data Loading** | 7 tables, 50M+ rows | Loaded all tables, checked shapes and memory |
| **EDA** | Target distribution, missing values, distributions | Identified 8% class imbalance, feature patterns |
| **Preprocessing** | DAYS_EMPLOYED fix, encoding, imputation | Created anomaly flag for 365243 placeholder |
| **Feature Engineering** | 5 table aggregations + domain features | 700+ features from multi-table joins |
| **Feature Selection** | Variance → Correlation → Importance | Reduced to top 400 features |
| **Modeling** | LightGBM CV, XGBoost, Logistic Regression | 5-fold StratifiedKFold with scale_pos_weight |
| **Evaluation** | AUC, KS, Gini, Business metrics, SHAP | Industry-standard credit risk KPIs |

---

### Model Performance

| Metric | Value | Benchmark |
|---|---|---|
| AUC-ROC | ~0.795 | >0.75 good |
| Gini | ~0.59 | >0.50 good |
| KS Statistic | ~0.38 | >0.30 acceptable |

---

### Headroom Dataset Summary

| Task ID | Stage | Failure Category | Difficulty |
|---|---|---|---|
| HT-001 | Preprocessing | domain_knowledge | hard |
| HT-002 | Feature Engineering | data_leakage | hard |
| HT-003 | Feature Engineering | feature_engineering_strategy | hard |
| HT-004 | Feature Selection | domain_knowledge | hard |
| HT-005 | Modeling | model_selection | hard |
| HT-006 | Evaluation | metric_selection | hard |
| HT-007 | Evaluation | domain_knowledge | hard |
| HT-008 | Feature Engineering | feature_engineering_strategy | medium |
| HT-009 | Modeling | model_selection | medium |
| HT-010 | Preprocessing | data_leakage | medium |

**Dominant failure categories:** domain_knowledge (credit risk specific), data_leakage (temporal & structural), model_selection (production constraints), metric_selection (business context)

---

### Key Findings

1. **DAYS_EMPLOYED anomaly (365243)** — domain-specific knowledge; LLMs without credit domain knowledge will misclassify this as an outlier.
2. **Threshold ≠ 0.5** — with 8% base rate, 0.5 threshold is inappropriate; optimal threshold is ~0.10–0.15 under a 10:1 cost ratio.
3. **Credit risk metrics** — KS and Gini are industry standards not present in standard ML curricula; LLMs default to accuracy/F1.
4. **Regulatory constraints** — age (DAYS_BIRTH) is a strong predictor but faces fair lending legal constraints; LLMs do not raise this.
5. **Structural missingness** — no-bureau-history NaN is not random missing data; imputation strategy requires domain reasoning.

---

*Notebook generated for end-to-end ML project + headroom dataset construction.*